# 基於故障可預測性分析之銑床兩階段預測性維護決策支援系統

**AI4I 2020 Predictive Maintenance Dataset — 完整實驗 Notebook**

## 執行順序
1. 資料準備與特徵工程
2. 實驗A：故障可診斷性量化（1-vs-Rest）
3. 實驗B：特徵工程消融實驗（Ablation Study）
4. Stage 1：排除 RNF，訓練 9 個模型（含 TWF）
5. Stage 2：排除 TWF & RNF，訓練第二階段模型
6. SHAP 可解釋性分析
7. TabNet 深度學習模型
8. 圖表產生（模型比較、Deep Analysis、三實驗報告）

## 套件需求
```
pip install scikit-learn xgboost lightgbm imbalanced-learn pytorch-tabnet torch shap plotly pandas numpy matplotlib
```

## 資料集
請將 `ai4i2020.csv` 放在與此 notebook 相同的目錄，或修改各 cell 中的 `DATA_PATH`。

In [2]:
# Colab 進行matplotlib繪圖時顯示繁體中文
# 下載台北思源黑體並命名taipei_sans_tc_beta.ttf，移至指定路徑
!wget -O TaipeiSansTCBeta-Regular.ttf https://drive.google.com/uc?id=1eGAsTN1HBpJAkeVM57_C7ccp7hbgSz3_&export=download

import matplotlib

# 改style要在改font之前
# plt.style.use('seaborn')

matplotlib.font_manager.fontManager.addfont('TaipeiSansTCBeta-Regular.ttf')
matplotlib.rc('font', family='Taipei Sans TC Beta')
FONT_PATH = '/content/TaipeiSansTCBeta-Regular.ttf'

--2026-05-11 16:28:33--  https://drive.google.com/uc?id=1eGAsTN1HBpJAkeVM57_C7ccp7hbgSz3_
Resolving drive.google.com (drive.google.com)... 74.125.68.138, 74.125.68.113, 74.125.68.139, ...
Connecting to drive.google.com (drive.google.com)|74.125.68.138|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: https://drive.usercontent.google.com/download?id=1eGAsTN1HBpJAkeVM57_C7ccp7hbgSz3_ [following]
--2026-05-11 16:28:33--  https://drive.usercontent.google.com/download?id=1eGAsTN1HBpJAkeVM57_C7ccp7hbgSz3_
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 172.253.118.132, 2404:6800:4003:c04::84
Connecting to drive.usercontent.google.com (drive.usercontent.google.com)|172.253.118.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 20659344 (20M) [application/octet-stream]
Saving to: ‘TaipeiSansTCBeta-Regular.ttf’

TaipeiSansTCBeta-Re 100%[===================>]  19.70M  --.-KB/s    in 0.1s    

2026-05-11 16:28:35

## 0. 安裝套件（Google Colab 請執行此 cell）

In [3]:
# Google Colab 安裝套件
!pip install xgboost lightgbm imbalanced-learn pytorch-tabnet shap --quiet

# Google Drive 掛載（如資料放在 Drive）
from google.colab import drive
drive.mount('/content/drive')
DATA_PATH = '/content/drive/MyDrive/115專題/ai4i_complete_code/ai4i2020.csv'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 1.5 MB/s eta 0:00:00
Mounted at /content/drive


## 1. 資料準備與特徵工程

讀取 AI4I 2020 資料集，衍生 4 個領域知識特徵：
- **Power**：功率 = 轉速 × 扭矩（對應 PWF 觸發條件）
- **Power wear**：功率×磨耗（對應 OSF 觸發條件）
- **Temperature difference**：溫差（對應 HDF 觸發條件）
- **Temperature power**：溫差/功率（複合特徵）

In [4]:
"""
01_data_preparation.py
資料讀取、清理、特徵工程
輸出：df（全局 DataFrame）、FEAT_COLS、FAILURE_COLS
"""
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# --- 讀取與基本清理 ---
df = pd.read_csv(DATA_PATH)
df.drop(['UDI', 'Product ID'], axis=1, inplace=True)
# Machine failure 是複合旗標，建模時用子旗標（TWF/HDF/PWF/OSF/RNF）即可
# 保留備用，但後續 FEAT_COLS 不含此欄

# --- 特徵工程(衍生特徵) ---
df['Power'] = df['Rotational speed [rpm]'] * df['Torque [Nm]']
df['Power wear'] = df['Power'] * df['Tool wear [min]']
df['Temperature difference'] = df['Process temperature [K]'] - df['Air temperature [K]']
df['Temperature power'] = np.where(
    df['Power'] != 0,
    df['Temperature difference'] / df['Power'], 0.0)

# --- One-hot 編碼（drop_first=True：Type_H 以 L=0, M=0 表示）---
df = pd.get_dummies(df, columns=['Type'], drop_first=True)
for c in df.columns:
    if df[c].dtype == bool:
        df[c] = df[c].astype(int)

FEAT_COLS = ['Air temperature [K]', 'Process temperature [K]',
             'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]',
             'Power', 'Power wear', 'Temperature difference',
             'Temperature power', 'Type_L', 'Type_M']

FAILURE_COLS = ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']
FAILURE_NATURE = {
    'TWF': '半隨機型', 'HDF': '規則型', 'PWF': '規則型',
    'OSF': '規則型',  'RNF': '純隨機型',
}

print("資料集形狀：", df.shape)
print(f"特徵欄位數：{len(FEAT_COLS)}")
print("\n各故障樣本數：")
for fc in FAILURE_COLS:
    n = df[fc].sum()
    print(f"  {fc}（{FAILURE_NATURE[fc]}）：{n} 筆 ({n/len(df):.2%})")

# 確認無 NaN / inf
assert df[FEAT_COLS].isnull().sum().sum() == 0, "有缺值！"
assert not np.isinf(df[FEAT_COLS].values).any(), "有 inf！"
print("\n✓ 無缺值、無 inf，資料準備完成")
print("✓ 全局變數已建立：df, FEAT_COLS, FAILURE_COLS, FAILURE_NATURE")

資料集形狀： (10000, 17)
特徵欄位數：11

各故障樣本數：
  TWF（半隨機型）：46 筆 (0.46%)
  HDF（規則型）：115 筆 (1.15%)
  PWF（規則型）：95 筆 (0.95%)
  OSF（規則型）：98 筆 (0.98%)
  RNF（純隨機型）：19 筆 (0.19%)

✓ 無缺值、無 inf，資料準備完成
✓ 全局變數已建立：df, FEAT_COLS, FAILURE_COLS, FAILURE_NATURE


## 2. 實驗A：故障可診斷性量化（1-vs-Rest）

對每種故障類型分別進行 1-vs-Rest 二元分類（XGBoost + 5-Fold CV），
量化各故障的 AUC、Average Precision、Recall、MCC。

**預期結果**：
- HDF/PWF/OSF（規則型）：AUC > 0.999
- TWF（半隨機型）：AUC ≈ 0.964，有限預測力
- RNF（純隨機型）：AUC ≈ 0.663，等同隨機猜測

In [5]:
# ── 實驗 A：各故障類型可診斷性量化（1-vs-Rest）─────────────────────
# 使用 Cell 1 已建立的 df, FEAT_COLS, FAILURE_COLS, FAILURE_NATURE
# 不重複做特徵工程或 get_dummies

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (roc_auc_score, average_precision_score,
                matthews_corrcoef, recall_score)
from xgboost import XGBClassifier

FAILURE_NAMES = {
    'TWF': '刀具磨耗故障 (Tool Wear)',
    'HDF': '散熱不良故障 (Heat Dissipation)',
    'PWF': '功率異常故障 (Power Failure)',
    'OSF': '過負荷故障 (Overstrain)',
    'RNF': '隨機故障 (Random)',
}

X = df[FEAT_COLS].astype(float).values

print("=" * 65)
print("【實驗 A】各故障類型可診斷性分析（1-vs-Rest，XGBoost 5-Fold）")
print("=" * 65)
print(f"{'故障':8s} {'本質':8s} {'樣本數':6s} {'正例率':6s} "
      f"{'AUC':7s} {'AP':7s} {'Recall':7s} {'MCC':7s}")
print("-" * 65)

rows_a = []
for fc in FAILURE_COLS:
    y     = df[fc].values.astype(int)
    n_pos = y.sum()
    ratio = y.mean()

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
    aucs, aps, recs, mccs = [], [], [], []

    for tr, te in skf.split(X, y):
        sc  = MinMaxScaler()
        Xtr = sc.fit_transform(X[tr])
        Xte = sc.transform(X[te])
        ytr, yte = y[tr], y[te]

        mdl = XGBClassifier(
            n_estimators=100, max_depth=5, learning_rate=0.1,
            eval_metric='logloss', random_state=0, n_jobs=-1,
            scale_pos_weight=(ytr == 0).sum() / max(ytr.sum(), 1))
        mdl.fit(Xtr, ytr)
        yp  = mdl.predict(Xte)
        ypr = mdl.predict_proba(Xte)[:, 1]

        if yte.sum() > 0:
            aucs.append(roc_auc_score(yte, ypr))
            aps.append(average_precision_score(yte, ypr))
            recs.append(recall_score(yte, yp, zero_division=0))
            mccs.append(matthews_corrcoef(yte, yp))

    auc_m = np.mean(aucs)
    ap_m  = np.mean(aps)
    rec_m = np.mean(recs)
    mcc_m = np.mean(mccs)
    nature = FAILURE_NATURE[fc]

    print(f"{fc:8s} {nature:8s} {n_pos:6d} {ratio:6.2%} "
          f"{auc_m:7.4f} {ap_m:7.4f} {rec_m:7.4f} {mcc_m:7.4f}")

    rows_a.append(dict(
        故障代碼=fc,
        故障名稱=FAILURE_NAMES[fc],
        本質類型=nature,
        樣本數=n_pos,
        正例率=round(ratio, 4),
        AUC=round(auc_m, 4),
        AP=round(ap_m, 4),
        Recall=round(rec_m, 4),
        MCC=round(mcc_m, 4),
    ))

df_a = pd.DataFrame(rows_a)

# 儲存到 Drive（runtime 重啟不會消失）
SAVE_DIR = '/content/drive/MyDrive/115專題/ai4i_complete_code/'
df_a.to_csv(SAVE_DIR + 'expA_diagnostics.csv', index=False, encoding='utf-8-sig')
print("\n✓ 儲存 expA_diagnostics.csv →", SAVE_DIR)

print("\n【決策建議】")
for _, row in df_a.iterrows():
    auc = row['AUC']
    if auc >= 0.99:
        dec = "✅ 規則型 → 納入 Stage 2 ML"
    elif auc >= 0.80:
        dec = "⚠️  半隨機型 → Stage 1 規則警示"
    else:
        dec = "❌ 純隨機型 → 排除，不建模"
    print(f"  {row['故障代碼']}（AUC={auc:.4f}）→ {dec}")

【實驗 A】各故障類型可診斷性分析（1-vs-Rest，XGBoost 5-Fold）
故障       本質       樣本數    正例率    AUC     AP      Recall  MCC    
-----------------------------------------------------------------
TWF      半隨機型         46  0.46%  0.9637  0.1088  0.1733  0.1153
HDF      規則型         115  1.15%  1.0000  0.9966  0.9652  0.9570
PWF      規則型          95  0.95%  0.9992  0.9092  0.9684  0.8788
OSF      規則型          98  0.98%  0.9998  0.9853  0.9795  0.9256
RNF      純隨機型         19  0.19%  0.6629  0.0051  0.0000 -0.0018

✓ 儲存 expA_diagnostics.csv → /content/drive/MyDrive/115專題/ai4i_complete_code/

【決策建議】
  TWF（AUC=0.9637）→ ⚠️  半隨機型 → Stage 1 規則警示
  HDF（AUC=1.0000）→ ✅ 規則型 → 納入 Stage 2 ML
  PWF（AUC=0.9992）→ ✅ 規則型 → 納入 Stage 2 ML
  OSF（AUC=0.9998）→ ✅ 規則型 → 納入 Stage 2 ML
  RNF（AUC=0.6629）→ ❌ 純隨機型 → 排除，不建模


## 2b：TWF 可預測性分析
- TWF：故障集中在 Tool wear > 200 min，但發生率低且不穩定。屬於條件隨機事件 → Stage 1 保留警示，Stage 2 排除

In [6]:
# ── Cell 2b：TWF 可預測性分析────────────────────────────────────────
# 透過資料分析驗證 TWF 的條件隨機本質，為 Stage 2 排除 TWF 提供依據
# 使用 Cell 1 已建立的 df

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

FONT_PATH = '/content/TaipeiSansTCBeta-Regular.ttf'
def fprop(size=10):
    return fm.FontProperties(fname=FONT_PATH, size=size)

plt.rcParams.update({
    'figure.facecolor': '#0d0d1a', 'axes.facecolor':  '#1a1a2e',
    'axes.edgecolor':   '#2a2a4a', 'axes.labelcolor': '#c0c8ff',
    'xtick.color':      '#8888aa', 'ytick.color':     '#8888aa',
    'text.color':       '#c0c8ff', 'grid.color':      '#2a2a4a',
    'grid.alpha': 0.4,
})

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor('#0d0d1a')
fig.suptitle('TWF 可預測性分析 — 條件隨機本質驗證',
             fontproperties=fprop(14), color='#e0e8ff', y=1.02)

# ── 分析1：Tool wear 分布（TWF=1 vs 正常）────────────────────────────
ax1 = axes[0]
wear_twf  = df[df['TWF'] == 1]['Tool wear [min]']
wear_norm = df[(df['TWF'] == 0) & (df['Machine failure'] == 0)]['Tool wear [min]']

ax1.hist(wear_norm, bins=40, color='#7bb4f7', alpha=0.5,
         label='正常樣本', density=True)
ax1.hist(wear_twf,  bins=20, color='#f87171', alpha=0.8,
         label='TWF 故障', density=True)
ax1.axvline(200, color='#f7c47b', linestyle='--', linewidth=1.5)
ax1.text(202, ax1.get_ylim()[1] * 0.85, '200 min',
         fontproperties=fprop(9), color='#f7c47b')
ax1.set_xlabel('Tool wear [min]', fontproperties=fprop(10))
ax1.set_ylabel('密度', fontproperties=fprop(10))
ax1.set_title('分析1：Tool wear 分布對比（TWF vs 正常）\n'
              '（TWF 幾乎全部集中在 200 min 以後）',
              fontproperties=fprop(11), color='#e0e0ff')
ax1.legend(prop=fprop(9))
ax1.grid(True, axis='y')

# ── 分析2：各磨耗區間的 TWF 發生率────────────────────────────────────
ax2 = axes[1]
ratios, centers = [], []
for lo in range(0, 260, 10):
    hi  = lo + 10
    sub = df[(df['Tool wear [min]'] >= lo) & (df['Tool wear [min]'] < hi)]
    if len(sub) >= 3:
        ratios.append(sub['TWF'].mean())
        centers.append(lo + 5)

ax2.bar(centers, ratios, width=9,
        color=['#f87171' if c >= 200 else '#7bb4f7' for c in centers],
        alpha=0.85, edgecolor='#0d0d1a')
ax2.axvline(200, color='#f7c47b', linestyle='--', linewidth=1.5)
ax2.text(202, max(ratios) * 0.95, '200 min 分界',
         fontproperties=fprop(8), color='#f7c47b')
ax2.set_xlabel('Tool wear 區間 [min]', fontproperties=fprop(10))
ax2.set_ylabel('TWF 發生率', fontproperties=fprop(10))
ax2.set_title('分析2：各磨耗區間的 TWF 發生率\n'
              '（200 min 後突增且各區間跳動不穩定）',
              fontproperties=fprop(11), color='#e0e0ff')
ax2.grid(True, axis='y')

plt.tight_layout()
plt.savefig(SAVE_DIR + 'fig_twf_analysis.png', dpi=150,
            bbox_inches='tight', facecolor='#0d0d1a')
plt.close()
print("✓ fig_twf_analysis.png 已儲存")

print("\n【TWF 各區間發生率】")
for lo in range(200, 260, 10):
    hi  = lo + 10
    sub = df[(df['Tool wear [min]'] >= lo) & (df['Tool wear [min]'] < hi)]
    if len(sub) >= 1:
        print(f"  {lo}-{hi} min：{sub['TWF'].sum():2d} / {len(sub):3d} 筆 "
              f"（發生率 {sub['TWF'].mean():.2%}）")

print("\n【結論】")
print("TWF 故障集中在 Tool wear > 200 min 的區間，")
print("但各區間發生率低且不穩定，無單調趨勢，")
print("屬於條件隨機事件 → Stage 2 排除。")

✓ fig_twf_analysis.png 已儲存

【TWF 各區間發生率】
  200-210 min：17 / 407 筆 （發生率 4.18%）
  210-220 min：12 / 253 筆 （發生率 4.74%）
  220-230 min： 9 /  96 筆 （發生率 9.38%）
  230-240 min： 5 /  31 筆 （發生率 16.13%）
  240-250 min： 1 /  12 筆 （發生率 8.33%）
  250-260 min： 1 /   2 筆 （發生率 50.00%）

【結論】
TWF 故障集中在 Tool wear > 200 min 的區間，
但各區間發生率低且不穩定，無單調趨勢，
屬於條件隨機事件 → Stage 2 排除。


## 2c：RNF 可預測性分析
- RNF：無論磨耗高低，發生率均接近全局平均值（0.19%）。無任何特徵可區分 → Stage 1 起即排除

In [7]:
# ── Cell 2c：RNF 可預測性分析────────────────────────────────────────
# 透過資料分析驗證 RNF 的純隨機本質，為 Stage 1 排除 RNF 提供依據
# 使用 Cell 1 已建立的 df

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from matplotlib.lines import Line2D

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor('#0d0d1a')
fig.suptitle('RNF 可預測性分析 — 純隨機本質驗證',
             fontproperties=fprop(14), color='#e0e8ff', y=1.02)

mask_norm = df['Machine failure'] == 0
mask_rnf  = df['RNF'] == 1

# ── 分析1：二維散佈圖（轉速 vs 扭矩）────────────────────────────────
ax1 = axes[0]

ax1.scatter(df.loc[mask_norm, 'Rotational speed [rpm]'],
            df.loc[mask_norm, 'Torque [Nm]'],
            s=3, alpha=0.15, color='#7bb4f7', label='正常樣本')
ax1.scatter(df.loc[mask_rnf, 'Rotational speed [rpm]'],
            df.loc[mask_rnf, 'Torque [Nm]'],
            s=80, alpha=0.9, color='#f87171',
            label=f'RNF 故障（{mask_rnf.sum()} 筆）',
            zorder=5, marker='X')
ax1.set_xlabel('轉速 [rpm]', fontproperties=fprop(10))
ax1.set_ylabel('扭矩 [Nm]', fontproperties=fprop(10))
ax1.set_title('分析1：轉速 vs 扭矩 散佈圖\n'
              '（RNF 散落於正常樣本中，無聚集區域）',
              fontproperties=fprop(11), color='#e0e0ff')
ax1.legend(prop=fprop(9))
ax1.grid(True)

# ── 分析2：各特徵分布對比（violin + RNF 散點）────────────────────────
ax2 = axes[1]

feat_compare = {
    '轉速\n[rpm]':   'Rotational speed [rpm]',
    '扭矩\n[Nm]':    'Torque [Nm]',
    '磨耗\n[min]':   'Tool wear [min]',
    '空氣溫度\n[K]': 'Air temperature [K]',
}

for i, (label, col) in enumerate(feat_compare.items()):
    nd   = df[mask_norm][col].values
    rd   = df[mask_rnf][col].values
    vmin, vmax = nd.min(), nd.max()
    nd_n = (nd - vmin) / (vmax - vmin)
    rd_n = (rd - vmin) / (vmax - vmin)

    vp = ax2.violinplot([nd_n], positions=[i - 0.2], widths=0.35,
                        showmedians=True)
    for pc in vp['bodies']:
        pc.set_facecolor('#7bb4f7')
        pc.set_alpha(0.5)
    for partname in ('cbars', 'cmins', 'cmaxes', 'cmedians'):
        vp[partname].set_color('#7bb4f7')

    ax2.scatter([i + 0.2] * len(rd_n), rd_n,
                s=60, color='#f87171', alpha=0.9,
                zorder=5, marker='X')

ax2.set_xticks(range(len(feat_compare)))
ax2.set_xticklabels(feat_compare.keys(), fontproperties=fprop(9))
ax2.set_ylabel('標準化特徵值（0-1）', fontproperties=fprop(10))
ax2.set_title('分析2：各特徵分布對比\n'
              '（RNF 散點落在正常分布範圍內，無異常聚集）',
              fontproperties=fprop(11), color='#e0e0ff')
ax2.legend(handles=[
    Line2D([0], [0], color='#7bb4f7', linewidth=6, alpha=0.5, label='正常樣本分布'),
    Line2D([0], [0], marker='X', color='#f87171', linewidth=0,
           markersize=8, label='RNF 故障'),
], prop=fprop(9))
ax2.grid(True, axis='y')

plt.tight_layout()
plt.savefig(SAVE_DIR + 'fig_rnf_analysis.png', dpi=150,
            bbox_inches='tight', facecolor='#0d0d1a')
plt.close()
print("✓ fig_rnf_analysis.png 已儲存")

print("\n【結論】")
print("RNF 在 PCA 投影與各特徵分布上均與正常樣本無法區分，")
print("無任何製程參數可預測其發生，屬於純隨機事件 → Stage 1 起即排除。")

✓ fig_rnf_analysis.png 已儲存

【結論】
RNF 在 PCA 投影與各特徵分布上均與正常樣本無法區分，
無任何製程參數可預測其發生，屬於純隨機事件 → Stage 1 起即排除。


## 3. 實驗B：特徵工程消融實驗（Ablation Study）

逐一移除各特徵，觀察 MCC 變化量（ΔMCC），驗證衍生特徵的貢獻。

In [8]:
# ── 實驗 B：特徵工程消融實驗（Ablation Study）───────────────────────
# 使用 Cell 1 已建立的 df, FEAT_COLS, FAILURE_COLS

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import roc_auc_score, matthews_corrcoef, recall_score
from xgboost import XGBClassifier

# 二元標籤：排除 RNF，只看可診斷故障（HDF/PWF/OSF/TWF）
y_bin = ((df['TWF'] + df['HDF'] + df['PWF'] + df['OSF']) > 0).astype(int).values

ORIG_FEATS = ['Air temperature [K]', 'Process temperature [K]',
              'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]',
              'Type_L', 'Type_M']
ENG_FEATS  = ['Power', 'Power wear', 'Temperature difference', 'Temperature power']
ALL_FEATS  = ORIG_FEATS + ENG_FEATS  # 與 FEAT_COLS 相同，顯式定義方便後面 remove

FEAT_MEANING = {
    'Power': '功率 = 轉速×扭矩（對應 PWF）',
    'Power wear': '功率×磨耗（對應 OSF）',
    'Temperature difference': '溫差（對應 HDF）',
    'Temperature power': '溫差/功率（複合特徵）',
}

def cv3_binary(feat_list):
    X = df[feat_list].astype(float).values
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=0)
    mc, au, re = [], [], []
    for tr, te in skf.split(X, y_bin):
        sc  = MinMaxScaler()
        Xtr = sc.fit_transform(X[tr])
        Xte = sc.transform(X[te])
        mdl = XGBClassifier(
            n_estimators=100, max_depth=5, learning_rate=0.1,
            eval_metric='logloss', random_state=0, n_jobs=-1)
        mdl.fit(Xtr, y_bin[tr])
        yp  = mdl.predict(Xte)
        ypr = mdl.predict_proba(Xte)[:, 1]
        mc.append(matthews_corrcoef(y_bin[te], yp))
        au.append(roc_auc_score(y_bin[te], ypr))
        re.append(recall_score(y_bin[te], yp, zero_division=0))
    return np.mean(mc), np.std(mc), np.mean(au), np.mean(re)

print("=" * 70)
print("【實驗 B】特徵工程消融實驗（Ablation Study，二元分類排除 RNF）")
print("=" * 70)
print(f"{'特徵組合':20s} \t{'特徵數'} \t{'MCC'} \t{'±std'} \t{'AUC'} \t{'Recall'}")
print("-" * 70)

rows_b = []

# 1. 完整特徵（基準）
mcc, sd, auc, rec = cv3_binary(ALL_FEATS)
baseline_mcc = mcc
print(f"{'完整特徵（基準）':20s} \t{len(ALL_FEATS)} \t{mcc:.4f} \t{sd:.4f} \t{auc:.4f} \t{rec:.4f}")
rows_b.append(dict(特徵組合='完整特徵（基準）', 特徵數=len(ALL_FEATS),
                   MCC=round(mcc,4), MCC_std=round(sd,4),
                   AUC=round(auc,4), Recall=round(rec,4), ΔMCC=0.0))

# 2. 只用原始特徵
mcc, sd, auc, rec = cv3_binary(ORIG_FEATS)
d = mcc - baseline_mcc
print(f"{'只用原始特徵（無衍生）':15s} \t{len(ORIG_FEATS)} \t{mcc:.4f} \t{sd:.4f} \t{auc:.4f} \t{rec:.4f} \tΔMCC={d:+.4f}")
rows_b.append(dict(特徵組合='只用原始特徵', 特徵數=len(ORIG_FEATS),
                   MCC=round(mcc,4), MCC_std=round(sd,4),
                   AUC=round(auc,4), Recall=round(rec,4), ΔMCC=round(d,4)))

# 3. 只用衍生特徵
mcc, sd, auc, rec = cv3_binary(ENG_FEATS)
d = mcc - baseline_mcc
print(f"{'只用衍生特徵':20s} \t{len(ENG_FEATS)} \t{mcc:.4f} \t{sd:.4f} \t{auc:.4f} \t{rec:.4f} \tΔMCC={d:+.4f}")
rows_b.append(dict(特徵組合='只用衍生特徵', 特徵數=len(ENG_FEATS),
                   MCC=round(mcc,4), MCC_std=round(sd,4),
                   AUC=round(auc,4), Recall=round(rec,4), ΔMCC=round(d,4)))

# 4. 逐一移除衍生特徵
print("\n── 逐一移除衍生特徵 ──")
for ef in ENG_FEATS:
    reduced = [f for f in ALL_FEATS if f != ef]
    mcc, sd, auc, rec = cv3_binary(reduced)
    d = mcc - baseline_mcc
    print(f"  移除 {ef:30s} MCC={mcc:.4f}  ΔMCC={d:+.4f}  [{FEAT_MEANING[ef]}]")
    rows_b.append(dict(特徵組合=f'移除 {ef}', 特徵數=len(reduced),
                       MCC=round(mcc,4), MCC_std=round(sd,4),
                       AUC=round(auc,4), Recall=round(rec,4), ΔMCC=round(d,4)))

# 5. 逐一移除原始特徵
print("\n── 逐一移除原始特徵 ──")
for of in ORIG_FEATS:
    reduced = [f for f in ALL_FEATS if f != of]
    mcc, sd, auc, rec = cv3_binary(reduced)
    d = mcc - baseline_mcc
    print(f"  移除 {of:30s} MCC={mcc:.4f}  ΔMCC={d:+.4f}")
    rows_b.append(dict(特徵組合=f'移除 {of}', 特徵數=len(reduced),
                       MCC=round(mcc,4), MCC_std=round(sd,4),
                       AUC=round(auc,4), Recall=round(rec,4), ΔMCC=round(d,4)))

df_b = pd.DataFrame(rows_b)
df_b.to_csv(SAVE_DIR + 'expB_ablation.csv', index=False, encoding='utf-8-sig')
print("\n✓ 儲存 expB_ablation.csv →", SAVE_DIR)

【實驗 B】特徵工程消融實驗（Ablation Study，二元分類排除 RNF）
特徵組合                 	特徵數 	MCC 	±std 	AUC 	Recall
----------------------------------------------------------------------
完整特徵（基準）             	11 	0.8076 	0.0227 	0.9920 	0.7545
只用原始特徵（無衍生）     	7 	0.7193 	0.0286 	0.9857 	0.6242 	ΔMCC=-0.0883
只用衍生特徵               	4 	0.5816 	0.0343 	0.9744 	0.5121 	ΔMCC=-0.2259

── 逐一移除衍生特徵 ──
  移除 Power                          MCC=0.7882  ΔMCC=-0.0193  [功率 = 轉速×扭矩（對應 PWF）]
  移除 Power wear                     MCC=0.7969  ΔMCC=-0.0107  [功率×磨耗（對應 OSF）]
  移除 Temperature difference         MCC=0.7967  ΔMCC=-0.0109  [溫差（對應 HDF）]
  移除 Temperature power              MCC=0.8071  ΔMCC=-0.0005  [溫差/功率（複合特徵）]

── 逐一移除原始特徵 ──
  移除 Air temperature [K]            MCC=0.8098  ΔMCC=+0.0022
  移除 Process temperature [K]        MCC=0.8136  ΔMCC=+0.0061
  移除 Rotational speed [rpm]         MCC=0.7446  ΔMCC=-0.0630
  移除 Torque [Nm]                    MCC=0.8117  ΔMCC=+0.0041
  移除 Tool wear [min]                MCC=0.8280  ΔMCC=+0.0

## 4. 資料前處理
### 4a：Stage 1 資料前處理（排除 RNF）



In [9]:
# ── Cell 4a：Stage 1 資料前處理──────────────────────────────────────
# 排除條件：RNF（純隨機）；TWF 保留為獨立類別
# 使用 Cell 1 已建立的 df, FEAT_COLS

import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from imblearn.over_sampling import SMOTE

# 只排除 RNF
mask_keep = (df['RNF'] == 0)
df_s1 = df[mask_keep].copy()

print(f"原始資料：{len(df)} 筆")
print(f"排除 RNF：{df['RNF'].sum()} 筆")
print(f"保留資料：{len(df_s1)} 筆")

# 建立多元故障標籤
# 0=正常, 1=TWF, 2=HDF, 3=PWF, 4=OSF
label = np.zeros(len(df_s1), dtype=int)
label[df_s1['TWF'].values == 1] = 1
label[df_s1['HDF'].values == 1] = 2
label[df_s1['PWF'].values == 1] = 3
label[df_s1['OSF'].values == 1] = 4

print("\nStage 1 各類別分布：")
for cls, name in enumerate(['正常', 'TWF', 'HDF', 'PWF', 'OSF']):
    print(f"  Class {cls}（{name}）：{(label == cls).sum()} 筆")

X = df_s1[FEAT_COLS].astype(float)
y = pd.Series(label)

# 分層切割 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y)

# MinMaxScaler（只在訓練集 fit）
sc = MinMaxScaler()
X_train_sc = pd.DataFrame(sc.fit_transform(X_train), columns=FEAT_COLS)
X_test_sc  = pd.DataFrame(sc.transform(X_test),      columns=FEAT_COLS)

# 清理欄位名稱（XGBoost 不接受 [ ] 字元）
FEAT_COLS_SAFE = [c.replace('[', '').replace(']', '').replace('<', '').strip()
                  for c in FEAT_COLS]
X_train_sc.columns = FEAT_COLS_SAFE
X_test_sc.columns  = FEAT_COLS_SAFE

# SMOTE（只在訓練集，k=3 因少數類樣本少）
sm = SMOTE(random_state=0, k_neighbors=3)
X_tr_sm, y_tr_sm = sm.fit_resample(X_train_sc, y_train)
print(f"\nSMOTE 後訓練集各類：")
for cls, cnt in sorted(pd.Series(y_tr_sm).value_counts().items()):
    print(f"  Class {cls}：{cnt} 筆")

# 儲存
stage1_state = dict(
    X_train_sc=X_train_sc, X_test_sc=X_test_sc,
    X_tr_sm=X_tr_sm,       y_tr_sm=y_tr_sm,
    y_train=y_train,        y_test=y_test,
    feat_cols=FEAT_COLS_SAFE, scaler=sc,
)
with open(SAVE_DIR + 'state_stage1.pkl', 'wb') as f:
    pickle.dump(stage1_state, f)
print("\n✓ Cell 4a 完成，state_stage1.pkl 已儲存")

原始資料：10000 筆
排除 RNF：19 筆
保留資料：9981 筆

Stage 1 各類別分布：
  Class 0（正常）：9652 筆
  Class 1（TWF）：42 筆
  Class 2（HDF）：106 筆
  Class 3（PWF）：83 筆
  Class 4（OSF）：98 筆

SMOTE 後訓練集各類：
  Class 0：7721 筆
  Class 1：7721 筆
  Class 2：7721 筆
  Class 3：7721 筆
  Class 4：7721 筆

✓ Cell 4a 完成，state_stage1.pkl 已儲存


### 4b：Stage 1 模型訓練（9 個模型）

In [10]:
# ── Cell 4b：Stage 1 模型訓練（9 個模型）────────────────────────────
# 依賴 Cell 4a 的 stage1_state

import time
from sklearn.metrics import (matthews_corrcoef, f1_score, accuracy_score,
                roc_auc_score, recall_score, precision_score)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                StackingClassifier)
from sklearn.neural_network import MLPClassifier
from lightgbm import LGBMClassifier

Xtr  = stage1_state['X_tr_sm']
ytr  = stage1_state['y_tr_sm']
Xte  = stage1_state['X_test_sc']
yte  = stage1_state['y_test']

def evaluate(name, model, fit_params=None):
    t0 = time.time()
    model.fit(Xtr, ytr, **(fit_params or {}))
    t1 = time.time()
    yp = model.predict(Xte)
    acc = accuracy_score(yte, yp)
    mcc = matthews_corrcoef(yte, yp)
    f1w = f1_score(yte, yp, average='weighted', zero_division=0)
    rec = recall_score(yte, yp, average='macro', zero_division=0)
    pre = precision_score(yte, yp, average='weighted', zero_division=0)
    try:
        ypr = model.predict_proba(Xte)
        auc = roc_auc_score(yte, ypr, multi_class='ovr', average='macro')
    except:
        auc = float('nan')
    tag = '★ DL' if any(k in name for k in ['MLP', 'Stacking']) else '  ML'
    print(f"{tag} [{name:30s}]  MCC={mcc:.4f}  F1={f1w:.4f}  "
          f"AUC={auc:.4f}  train={t1-t0:.1f}s")
    return model, dict(Accuracy=acc, MCC=mcc, F1w=f1w,
                       Recall_macro=rec, Precision=pre, AUC=auc,
                       Train_s=round(t1-t0, 3))

results1, models1 = {}, {}

print("=" * 70)
print("【Stage 1】傳統 ML 模型（排除 RNF，SMOTE 平衡，5 類）")
print("=" * 70)

for name, mdl in [
    ('KNN',               KNeighborsClassifier(n_neighbors=3)),
    ('Decision Tree',     DecisionTreeClassifier(random_state=0)),
    ('Random Forest',     RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=0)),
    ('Gradient Boosting', GradientBoostingClassifier(n_estimators=100, random_state=0)),
    ('XGBoost',           XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1,
                                        eval_metric='mlogloss', random_state=0, n_jobs=-1)),
    ('LightGBM',          LGBMClassifier(n_estimators=100, max_depth=6, learning_rate=0.1,
                                         random_state=0, n_jobs=-1, verbose=-1)),
]:
    models1[name], results1[name] = evaluate(name, mdl)

print("\n" + "=" * 70)
print("【Stage 1】深度學習模型")
print("=" * 70)

for name, mdl in [
    ('MLP (original)', MLPClassifier(hidden_layer_sizes=(25, 25, 25), activation='relu',
                                     solver='adam', batch_size=2000,
                                     max_iter=200, random_state=0)),
    ('MLP (upgraded)', MLPClassifier(hidden_layer_sizes=(128, 64, 32), activation='relu',
                                     solver='adam', alpha=1e-4, batch_size=256,
                                     learning_rate='adaptive', max_iter=300, random_state=0)),
]:
    models1[name], results1[name] = evaluate(name, mdl)

print("\n" + "=" * 70)
print("【Stage 1】Stacking Ensemble（XGB + LGBM → RF）")
print("=" * 70)

stack = StackingClassifier(
    estimators=[
        ('xgb',  XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1,
                               eval_metric='mlogloss', random_state=0, n_jobs=-1)),
        ('lgbm', LGBMClassifier(n_estimators=100, max_depth=5, learning_rate=0.1,
                                random_state=0, n_jobs=-1, verbose=-1)),
    ],
    final_estimator=RandomForestClassifier(n_estimators=50, class_weight='balanced',
                                           random_state=0, n_jobs=-1),
    cv=3, n_jobs=1, passthrough=False,
)
models1['Stacking (XGB+LGBM→RF)'], results1['Stacking (XGB+LGBM→RF)'] = evaluate(
    'Stacking (XGB+LGBM→RF)', stack)

# 儲存模型與結果
with open(SAVE_DIR + 'results_stage1.pkl', 'wb') as f:
    pickle.dump(dict(results=results1, models=models1,
                     Xte=Xte, yte=np.array(yte), feat=FEAT_COLS_SAFE), f)

df_r1 = pd.DataFrame(results1).T
df_r1.to_csv(SAVE_DIR + 'results_stage1.csv', encoding='utf-8-sig')
print("\n✓ 全部模型訓練完成")
print(df_r1[['MCC', 'F1w', 'AUC', 'Train_s']].round(4).to_string())

【Stage 1】傳統 ML 模型（排除 RNF，SMOTE 平衡，5 類）
  ML [KNN                           ]  MCC=0.5179  F1=0.9613  AUC=0.8760  train=0.1s
  ML [Decision Tree                 ]  MCC=0.7716  F1=0.9858  AUC=0.9162  train=0.8s
  ML [Random Forest                 ]  MCC=0.7837  F1=0.9867  AUC=0.9813  train=27.6s
  ML [Gradient Boosting             ]  MCC=0.7112  F1=0.9807  AUC=0.9944  train=145.3s
  ML [XGBoost                       ]  MCC=0.7406  F1=0.9833  AUC=0.9943  train=3.0s
  ML [LightGBM                      ]  MCC=0.8044  F1=0.9879  AUC=0.9953  train=1.7s

【Stage 1】深度學習模型
★ DL [MLP (original)                ]  MCC=0.5765  F1=0.9599  AUC=0.9913  train=34.5s
★ DL [MLP (upgraded)                ]  MCC=0.6911  F1=0.9793  AUC=0.9923  train=109.8s

【Stage 1】Stacking Ensemble（XGB + LGBM → RF）
★ DL [Stacking (XGB+LGBM→RF)        ]  MCC=0.8716  F1=0.9923  AUC=0.9418  train=17.6s

✓ 全部模型訓練完成
                           MCC     F1w     AUC  Train_s
KNN                     0.5179  0.9613  0.8760    0.075
Dec

## 5. Stage 2：排除 TWF & RNF，訓練第二階段模型
### 5a：Stage 2 資料前處理

In [11]:
# ── Cell 5a：Stage 2 資料前處理（排除 TWF & RNF，只保留規則型故障）
# 使用 Cell 1 已建立的 df, FEAT_COLS, FEAT_COLS_SAFE

# 只保留 HDF/PWF/OSF 與正常樣本（排除 TWF 和 RNF）
mask_s2 = (df['TWF'] == 0) & (df['RNF'] == 0)
df_s2   = df[mask_s2].copy()

label2 = np.zeros(len(df_s2), dtype=int)
label2[df_s2['HDF'].values == 1] = 1
label2[df_s2['PWF'].values == 1] = 2
label2[df_s2['OSF'].values == 1] = 3

print("Stage 2 各類別分布（排除 TWF & RNF）：")
for cls, name in enumerate(['正常', 'HDF', 'PWF', 'OSF']):
    print(f"  Class {cls}（{name}）：{(label2 == cls).sum()} 筆")
print(f"  總樣本數：{len(df_s2)} 筆")

X2 = df_s2[FEAT_COLS].astype(float)
y2 = pd.Series(label2)

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, random_state=0, stratify=y2)

sc2 = MinMaxScaler()
X2_train_sc = pd.DataFrame(sc2.fit_transform(X2_train), columns=FEAT_COLS_SAFE)
X2_test_sc  = pd.DataFrame(sc2.transform(X2_test),      columns=FEAT_COLS_SAFE)

sm2 = SMOTE(random_state=0, k_neighbors=3)
X2_tr_sm, y2_tr_sm = sm2.fit_resample(X2_train_sc, y2_train)
print(f"\nSMOTE 後訓練集各類：")
for cls, cnt in sorted(pd.Series(y2_tr_sm).value_counts().items()):
    print(f"  Class {cls}：{cnt} 筆")

stage2_state = dict(
    X_train_sc=X2_train_sc, X_test_sc=X2_test_sc,
    X_tr_sm=X2_tr_sm,       y_tr_sm=y2_tr_sm,
    y_train=y2_train,        y_test=y2_test,
    feat_cols=FEAT_COLS_SAFE, scaler=sc2,
)
with open(SAVE_DIR + 'state_stage2.pkl', 'wb') as f:
    pickle.dump(stage2_state, f)
print("\n✓ Cell 5a 完成，state_stage2.pkl 已儲存")

Stage 2 各類別分布（排除 TWF & RNF）：
  Class 0（正常）：9652 筆
  Class 1（HDF）：106 筆
  Class 2（PWF）：83 筆
  Class 3（OSF）：95 筆
  總樣本數：9936 筆

SMOTE 後訓練集各類：
  Class 0：7721 筆
  Class 1：7721 筆
  Class 2：7721 筆
  Class 3：7721 筆

✓ Cell 5a 完成，state_stage2.pkl 已儲存


### 5b：Stage 2 模型訓練

In [12]:
# ── Cell 5b：Stage 2 模型訓練（排除 TWF & RNF，4 類）────────────────
# Class 0（正常）、Class 1（HDF）、Class 2（PWF）、Class 3（OSF）
# 依賴 Cell 5a 的 stage2_state

Xtr2 = stage2_state['X_tr_sm']
ytr2 = stage2_state['y_tr_sm']
Xte2 = stage2_state['X_test_sc']
yte2 = stage2_state['y_test']

# evaluate 函式沿用 Cell 4b 定義，但資料換成 Stage 2
def evaluate2(name, model, fit_params=None):
    t0 = time.time()
    model.fit(Xtr2, ytr2, **(fit_params or {}))
    t1 = time.time()
    yp = model.predict(Xte2)
    acc = accuracy_score(yte2, yp)
    mcc = matthews_corrcoef(yte2, yp)
    f1w = f1_score(yte2, yp, average='weighted', zero_division=0)
    rec = recall_score(yte2, yp, average='macro', zero_division=0)
    pre = precision_score(yte2, yp, average='weighted', zero_division=0)
    try:
        ypr = model.predict_proba(Xte2)
        auc = roc_auc_score(yte2, ypr, multi_class='ovr', average='macro')
    except:
        auc = float('nan')
    tag = '★ DL' if any(k in name for k in ['MLP', 'Stacking']) else '  ML'
    print(f"{tag} [{name:30s}]  MCC={mcc:.4f}  F1={f1w:.4f}  "
          f"AUC={auc:.4f}  train={t1-t0:.1f}s")
    return model, dict(Accuracy=acc, MCC=mcc, F1w=f1w,
                       Recall_macro=rec, Precision=pre, AUC=auc,
                       Train_s=round(t1-t0, 3))

results2, models2 = {}, {}

print("=" * 70)
print("【Stage 2】傳統 ML 模型（排除 TWF & RNF，SMOTE 平衡，4 類）")
print("=" * 70)

for name, mdl in [
    ('KNN',               KNeighborsClassifier(n_neighbors=3)),
    ('Decision Tree',     DecisionTreeClassifier(random_state=0)),
    ('Random Forest',     RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=0)),
    ('Gradient Boosting', GradientBoostingClassifier(n_estimators=100, random_state=0)),
    ('XGBoost',           XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1,
                                        eval_metric='mlogloss', random_state=0, n_jobs=-1)),
    ('LightGBM',          LGBMClassifier(n_estimators=100, max_depth=6, learning_rate=0.1,
                                         random_state=0, n_jobs=-1, verbose=-1)),
]:
    models2[name], results2[name] = evaluate2(name, mdl)

print("\n" + "=" * 70)
print("【Stage 2】深度學習模型")
print("=" * 70)

for name, mdl in [
    ('MLP (original)', MLPClassifier(hidden_layer_sizes=(25, 25, 25), activation='relu',
                                     solver='adam', batch_size=2000,
                                     max_iter=200, random_state=0)),
    ('MLP (upgraded)', MLPClassifier(hidden_layer_sizes=(128, 64, 32), activation='relu',
                                     solver='adam', alpha=1e-4, batch_size=256,
                                     learning_rate='adaptive', max_iter=300, random_state=0)),
]:
    models2[name], results2[name] = evaluate2(name, mdl)

print("\n" + "=" * 70)
print("【Stage 2】Stacking Ensemble（XGB + LGBM → RF）")
print("=" * 70)

stack2 = StackingClassifier(
    estimators=[
        ('xgb',  XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1,
                               eval_metric='mlogloss', random_state=0, n_jobs=-1)),
        ('lgbm', LGBMClassifier(n_estimators=100, max_depth=5, learning_rate=0.1,
                                random_state=0, n_jobs=-1, verbose=-1)),
    ],
    final_estimator=RandomForestClassifier(n_estimators=50, class_weight='balanced',
                                           random_state=0, n_jobs=-1),
    cv=3, n_jobs=1, passthrough=False,
)
models2['Stacking (XGB+LGBM→RF)'], results2['Stacking (XGB+LGBM→RF)'] = evaluate2(
    'Stacking (XGB+LGBM→RF)', stack2)

with open(SAVE_DIR + 'results_stage2.pkl', 'wb') as f:
    pickle.dump(dict(results=results2, models=models2,
                     Xte=Xte2, yte=np.array(yte2), feat=FEAT_COLS_SAFE), f)

df_r2 = pd.DataFrame(results2).T
df_r2.to_csv(SAVE_DIR + 'results_stage2.csv', encoding='utf-8-sig')
print("\n✓ Stage 2 全部模型訓練完成")
print(df_r2[['MCC', 'F1w', 'AUC', 'Train_s']].round(4).to_string())

【Stage 2】傳統 ML 模型（排除 TWF & RNF，SMOTE 平衡，4 類）
  ML [KNN                           ]  MCC=0.5361  F1=0.9695  AUC=0.8584  train=0.1s
  ML [Decision Tree                 ]  MCC=0.8982  F1=0.9944  AUC=0.9260  train=0.3s
  ML [Random Forest                 ]  MCC=0.8910  F1=0.9940  AUC=0.9992  train=7.9s
  ML [Gradient Boosting             ]  MCC=0.9247  F1=0.9957  AUC=0.9995  train=69.9s
  ML [XGBoost                       ]  MCC=0.9106  F1=0.9951  AUC=0.9998  train=3.8s
  ML [LightGBM                      ]  MCC=0.9291  F1=0.9960  AUC=0.9997  train=1.3s

【Stage 2】深度學習模型
★ DL [MLP (original)                ]  MCC=0.7500  F1=0.9840  AUC=0.9968  train=24.2s
★ DL [MLP (upgraded)                ]  MCC=0.7958  F1=0.9871  AUC=0.9977  train=43.7s

【Stage 2】Stacking Ensemble（XGB + LGBM → RF）
★ DL [Stacking (XGB+LGBM→RF)        ]  MCC=0.9163  F1=0.9953  AUC=0.9761  train=12.3s

✓ Stage 2 全部模型訓練完成
                           MCC     F1w     AUC  Train_s
KNN                     0.5361  0.9695  0.8584  

## 6. SHAP 可解釋性分析

對 XGBoost 的每個故障類別計算 SHAP 值，驗證模型決策是否符合物理規則：
- HDF → 預期主特徵：溫差
- PWF → 預期主特徵：功率
- OSF → 預期主特徵：功率×磨耗

In [13]:
# ── Cell 6：SHAP 可解釋性分析（Stage 2，XGBoost）────────────────────
# 依賴 Cell 5b 的 stage2_state、results2、models2

import shap

# Stage 2 類別對應（0=正常, 1=HDF, 2=PWF, 3=OSF）
FAILURE_ZH_S2 = {0: '正常', 1: 'HDF 散熱不良', 2: 'PWF 功率異常', 3: 'OSF 過負荷'}
PHYSICS_TOP1  = {'HDF': '溫差', 'PWF': '功率', 'OSF': '功率×磨耗'}

# FEAT_COLS_SAFE 順序對應的中文短名
FEAT_SHORT = ['空氣溫度', '製程溫度', '轉速', '扭矩', '刀具磨耗',
              '功率', '功率×磨耗', '溫差', '溫差/功率', '類型L', '類型M']
assert len(FEAT_SHORT) == len(FEAT_COLS_SAFE), "FEAT_SHORT 長度與 FEAT_COLS_SAFE 不一致！"

Xtr2 = stage2_state['X_tr_sm'].values
ytr2 = np.array(stage2_state['y_tr_sm']).astype(int)
Xte2 = stage2_state['X_test_sc'].values
yte2 = np.array(stage2_state['y_test']).astype(int)

# 直接取 Cell 5b 已訓練好的 XGBoost，不重新 fit
xgb_s2 = models2['XGBoost']

print("=" * 60)
print("【實驗 C】SHAP 可解釋性分析（Stage 2 XGBoost）")
print("=" * 60)

exp = shap.TreeExplainer(xgb_s2)
shap_by_class = {}
rows_c = []

for cls in [1, 2, 3]:   # 只看三種故障類別（排除 Class 0 正常）
    mask = np.where(yte2 == cls)[0]
    if len(mask) == 0:
        print(f"  Class {cls}：測試集無樣本，跳過")
        continue

    X_cls  = Xte2[mask[:min(50, len(mask))]]   # 最多取 50 筆故障樣本
    sv_all = exp.shap_values(X_cls)             # shape: (n, f, n_classes)

    if sv_all.ndim == 3:
        sv_cls = sv_all[:, :, cls]
    else:
        sv_cls = sv_all

    mean_abs = np.abs(sv_cls).mean(axis=0)
    top3_idx = mean_abs.argsort()[::-1][:3]
    top3     = [(FEAT_SHORT[i], round(float(mean_abs[i]), 4)) for i in top3_idx]
    shap_by_class[cls] = top3

    name_zh = FAILURE_ZH_S2[cls]
    top3_str = '  |  '.join(f"{fn}={sv:.3f}" for fn, sv in top3)
    print(f"  {name_zh}：SHAP Top-3 → {top3_str}")

    rows_c.append({
        '故障': name_zh,
        'SHAP Top1': top3[0][0] if len(top3) > 0 else '',
        'SHAP Top2': top3[1][0] if len(top3) > 1 else '',
        'SHAP Top3': top3[2][0] if len(top3) > 2 else '',
        'SHAP Top1 值': top3[0][1] if len(top3) > 0 else 0,
    })

# 一致性驗證
print("\n【SHAP vs 物理規則 一致性驗證】")
physics_map = {'HDF 散熱不良': '溫差', 'PWF 功率異常': '功率', 'OSF 過負荷': '功率×磨耗'}
for row in rows_c:
    expected = physics_map.get(row['故障'], '無')
    top3_names = [row['SHAP Top1'], row['SHAP Top2'], row['SHAP Top3']]
    match = '✅ 一致' if expected in top3_names else '❌ 不一致'
    print(f"  {row['故障']}：物理預期={expected:8s}  SHAP Top1={row['SHAP Top1']:8s}  → {match}")

# XGBoost 內建特徵重要性
xgb_fi = pd.Series(xgb_s2.feature_importances_, index=FEAT_SHORT)
print("\nXGBoost 內建特徵重要性 Top-5：")
print(xgb_fi.sort_values(ascending=False).head(5).to_string())

# 儲存
df_c = pd.DataFrame(rows_c)
df_c.to_csv(SAVE_DIR + 'expC_shap.csv', index=False, encoding='utf-8-sig')

with open(SAVE_DIR + 'results_stage2.pkl', 'rb') as f:
    s2 = pickle.load(f)
s2['shap_by_class'] = shap_by_class
s2['xgb_fi']        = xgb_fi.to_dict()
s2['feat_short']    = FEAT_SHORT
with open(SAVE_DIR + 'results_stage2.pkl', 'wb') as f:
    pickle.dump(s2, f)

print("\n✓ expC_shap.csv 已儲存")
print("✓ SHAP 結果已更新至 results_stage2.pkl")

【實驗 C】SHAP 可解釋性分析（Stage 2 XGBoost）
  HDF 散熱不良：SHAP Top-3 → 溫差=4.141  |  功率×磨耗=0.598  |  轉速=0.271
  PWF 功率異常：SHAP Top-3 → 功率=4.011  |  刀具磨耗=1.198  |  扭矩=0.676
  OSF 過負荷：SHAP Top-3 → 功率×磨耗=4.257  |  類型L=0.438  |  扭矩=0.357

【SHAP vs 物理規則 一致性驗證】
  HDF 散熱不良：物理預期=溫差        SHAP Top1=溫差        → ✅ 一致
  PWF 功率異常：物理預期=功率        SHAP Top1=功率        → ✅ 一致
  OSF 過負荷：物理預期=功率×磨耗     SHAP Top1=功率×磨耗     → ✅ 一致

XGBoost 內建特徵重要性 Top-5：
功率       0.313317
溫差       0.270705
功率×磨耗    0.196449
轉速       0.107964
刀具磨耗     0.048508

✓ expC_shap.csv 已儲存
✓ SHAP 結果已更新至 results_stage2.pkl


## 7. TabNet 深度學習模型

使用 pytorch-tabnet 訓練 TabNet 模型，獲取 Sparse Attention 特徵重要性，
用於與 XGBoost SHAP 做跨架構比較。

**注意**：TabNet 訓練時間約 2 分鐘，記憶體需求較高。

**輸出**：`tabnet_result.pkl`

In [14]:
# ── Cell 7：TabNet 深度學習模型（Stage 2）───────────────────────────
# 依賴 Cell 5a 的 stage2_state、Cell 5b 的 results2、models2

from pytorch_tabnet.tab_model import TabNetClassifier
from sklearn.metrics import precision_score
import torch

Xtr2_np = stage2_state['X_tr_sm'].values.astype(np.float32)
ytr2_np = np.array(stage2_state['y_tr_sm']).astype(int)
Xte2_np = stage2_state['X_test_sc'].values.astype(np.float32)
yte2_np = np.array(stage2_state['y_test']).astype(int)

print("TabNet 訓練中（Stage 2，4 類）...")
t0 = time.time()
tn = TabNetClassifier(
    n_d=16, n_a=16, n_steps=3, gamma=1.5, lambda_sparse=1e-3,
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=2e-2),
    mask_type='sparsemax', verbose=0, seed=0)
tn.fit(Xtr2_np, ytr2_np,
       max_epochs=40, patience=10,
       batch_size=1024, virtual_batch_size=512)
t1 = time.time()

yp  = tn.predict(Xte2_np)
ypr = tn.predict_proba(Xte2_np)
acc = accuracy_score(yte2_np, yp)
mcc = matthews_corrcoef(yte2_np, yp)
f1w = f1_score(yte2_np, yp, average='weighted', zero_division=0)
rec = recall_score(yte2_np, yp, average='macro', zero_division=0)
pre = precision_score(yte2_np, yp, average='weighted', zero_division=0)
auc = roc_auc_score(yte2_np, ypr, multi_class='ovr', average='macro')

print(f"★ DL [{'TabNet':30s}]  MCC={mcc:.4f}  F1={f1w:.4f}  "
      f"AUC={auc:.4f}  train={t1-t0:.1f}s")

tn_fi = pd.Series(tn.feature_importances_, index=FEAT_SHORT)
print("\nTabNet 特徵重要性 Top-5：")
print(tn_fi.sort_values(ascending=False).head(5).to_string())

# 併入 results2 / models2，與其他模型統一管理
tabnet_result = dict(Accuracy=acc, MCC=mcc, F1w=f1w,
                     Recall_macro=rec, Precision=pre, AUC=auc,
                     Train_s=round(t1-t0, 3))
results2['TabNet'] = tabnet_result
models2['TabNet']  = tn

# 更新 results_stage2.pkl
with open(SAVE_DIR + 'results_stage2.pkl', 'rb') as f:
    s2 = pickle.load(f)
s2['results'] = results2
s2['models']  = models2
s2['tabnet_fi'] = tn_fi.to_dict()
with open(SAVE_DIR + 'results_stage2.pkl', 'wb') as f:
    pickle.dump(s2, f)

# 印出含 TabNet 的完整排行
df_r2 = pd.DataFrame(results2).T
print("\n【Stage 2 含 TabNet 完整排行】")
print(df_r2[['MCC', 'F1w', 'AUC', 'Train_s']].sort_values('MCC', ascending=False).round(4).to_string())
print("\n✓ TabNet 完成，results_stage2.pkl 已更新")

TabNet 訓練中（Stage 2，4 類）...
★ DL [TabNet                        ]  MCC=0.7842  F1=0.9873  AUC=0.9980  train=69.1s

TabNet 特徵重要性 Top-5：
功率×磨耗    0.253696
功率       0.191578
溫差       0.150833
扭矩       0.105462
轉速       0.102643

【Stage 2 含 TabNet 完整排行】
                           MCC     F1w     AUC  Train_s
LightGBM                0.9291  0.9960  0.9997    1.272
Gradient Boosting       0.9247  0.9957  0.9995   69.948
Stacking (XGB+LGBM→RF)  0.9163  0.9953  0.9761   12.335
XGBoost                 0.9106  0.9951  0.9998    3.797
Decision Tree           0.8982  0.9944  0.9260    0.350
Random Forest           0.8910  0.9940  0.9992    7.863
MLP (upgraded)          0.7958  0.9871  0.9977   43.725
TabNet                  0.7842  0.9873  0.9980   69.138
MLP (original)          0.7500  0.9840  0.9968   24.177
KNN                     0.5361  0.9695  0.8584    0.062

✓ TabNet 完成，results_stage2.pkl 已更新


## 8. 圖表：完整模型比較

產生 5 張研究用圖表：
1. 模型比較長條圖（MCC / AUC / F1）
2. 效率效能氣泡圖
3. 混淆矩陣
4. SHAP vs TabNet 特徵重要性對比
5. 研究結論摘要

**輸出**：fig1_model_comparison.png 等多個 PNG 檔

In [15]:
# ── Cell 8：圖表產生（完整模型比較）─────────────────────────────────
# 依賴 Cell 5b 的 results2、models2
#       Cell 5a 的 stage2_state
#       Cell 6  的 shap_by_class、xgb_fi（已存入 results_stage2.pkl）
#       Cell 7  的 results2['TabNet']、tn_fi

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from sklearn.metrics import confusion_matrix

FONT_PATH = '/content/TaipeiSansTCBeta-Regular.ttf'
def fprop(size=10):
    return fm.FontProperties(fname=FONT_PATH, size=size)

plt.rcParams.update({
    'figure.facecolor': '#0d0d1a', 'axes.facecolor':  '#1a1a2e',
    'axes.edgecolor':   '#2a2a4a', 'axes.labelcolor': '#c0c8ff',
    'xtick.color':      '#8888aa', 'ytick.color':     '#8888aa',
    'text.color':       '#c0c8ff', 'grid.color':      '#2a2a4a',
    'grid.alpha': 0.4,  'legend.facecolor': '#1a1a2e',
    'legend.edgecolor': '#2a2a4a',
})

# ── 載入資料（全從全局變數，不讀舊 pkl）──────────────────────────────
# 從 results_stage2.pkl 取回 SHAP 相關資料（Cell 6 存入）
with open(SAVE_DIR + 'results_stage2.pkl', 'rb') as f:
    s2 = pickle.load(f)
shap_by_class = s2['shap_by_class']   # key: 1=HDF, 2=PWF, 3=OSF
xgb_fi        = s2['xgb_fi']
tabnet_fi     = s2['tabnet_fi']

Xte2 = stage2_state['X_test_sc'].values
yte2 = np.array(stage2_state['y_test']).astype(int)

df_r = pd.DataFrame(results2).T[['MCC', 'F1w', 'AUC', 'Recall_macro', 'Train_s']]
df_r = df_r.astype(float)

# Stage 2 類別與顏色定義
ML_MODELS = ['KNN', 'Decision Tree', 'Random Forest',
             'Gradient Boosting', 'XGBoost', 'LightGBM']
DL_MODELS = ['MLP (original)', 'MLP (upgraded)', 'TabNet', 'Stacking (XGB+LGBM→RF)']

def model_color(name):
    return '#7bf7c8' if name in DL_MODELS else '#7bb4f7'

# Stage 2: 0=正常, 1=HDF, 2=PWF, 3=OSF
CLS_LABELS_S2  = ['正常', 'HDF', 'PWF', 'OSF']
FAILURE_MAP_S2 = {1: 'HDF 散熱不良', 2: 'PWF 功率異常', 3: 'OSF 過負荷'}
PHYSICS_S2     = {1: '溫差', 2: '功率', 3: '功率×磨耗'}
FAILURE_COLOR  = {
    'HDF 散熱不良': '#f97316',
    'PWF 功率異常': '#fbbf24',
    'OSF 過負荷':   '#a78bfa',
}

# ════════════════════════════════════════════════════════════════════════
# 圖1：完整模型比較（MCC / AUC / F1）— 橫條圖
# ════════════════════════════════════════════════════════════════════════
fig1, axes = plt.subplots(1, 3, figsize=(18, 7))
fig1.patch.set_facecolor('#0d0d1a')
fig1.suptitle('ML vs 深度學習 模型效能比較（Stage 2：排除 TWF & RNF）',
              fontproperties=fprop(13), color='#e0e8ff', y=1.02)

order   = ML_MODELS + DL_MODELS
df_plot = df_r.loc[[m for m in order if m in df_r.index]]
colors  = [model_color(m) for m in df_plot.index]

for ax, (col, title, xlim) in zip(axes, [
    ('MCC', 'MCC（主指標）',    (0.45, 1.00)),
    ('AUC', 'ROC-AUC macro',   (0.82, 1.01)),
    ('F1w', 'F1-Score（加權）', (0.95, 1.01)),
]):
    # 由高到低排序
    df_sorted = df_plot[[col]].copy()
    df_sorted['color'] = colors
    df_sorted = df_sorted.sort_values(col, ascending=True)  # ascending=True 讓最高的在上方

    bars = ax.barh(range(len(df_sorted)), df_sorted[col],
                   color=df_sorted['color'].tolist(),
                   edgecolor='#0d0d1a', linewidth=0.8, height=0.62)
    for i, (bar, v) in enumerate(zip(bars, df_sorted[col])):
        ax.text(v + 0.002, bar.get_y() + bar.get_height() / 2,
                f'{v:.4f}', va='center',
                fontproperties=fprop(8), color='#c0c8ff')
    ax.set_yticks(range(len(df_sorted)))
    ax.set_yticklabels(df_sorted.index, fontproperties=fprop(9))
    ax.set_xlim(xlim)
    ax.set_title(title, fontproperties=fprop(11), color='#e0e0ff')
    ax.grid(True, axis='x')

    # ML/DL 分界線（水平線，對應排序後的位置）
    ml_count = sum(1 for m in df_sorted.index if m in ML_MODELS)
    ax.axhline(y=ml_count - 0.5, color='#444466',
               linestyle='--', alpha=0.6, linewidth=1.2)

plt.tight_layout()
plt.savefig(SAVE_DIR + 'fig1_model_comparison.png', dpi=150,
            bbox_inches='tight', facecolor='#0d0d1a')
plt.close()
print("✓ 圖1")

# ════════════════════════════════════════════════════════════════════════
# 圖2：效能 vs 訓練效率氣泡圖 + Recall macro
# ════════════════════════════════════════════════════════════════════════
fig2, axes2 = plt.subplots(1, 2, figsize=(14, 6))
fig2.patch.set_facecolor('#0d0d1a')
fig2.suptitle('效能 vs 訓練效率（氣泡大小 = AUC）',
              fontproperties=fprop(13), color='#e0e8ff')

ax = axes2[0]
for name in df_plot.index:
    row = df_plot.loc[name]
    ax.scatter(row['Train_s'], row['MCC'],
               s=row['AUC'] * 300, color=model_color(name),
               alpha=0.85, edgecolors='white', linewidths=0.8, zorder=3)
    ax.annotate(name.replace(' (', '\n('), (row['Train_s'], row['MCC']),
                textcoords='offset points', xytext=(5, 5),
                fontproperties=fprop(7.5), color='#c0c8ff')
ax.set_xscale('log')
ax.set_xlabel('訓練時間（秒，對數）', fontproperties=fprop(10))
ax.set_ylabel('MCC', fontproperties=fprop(10))
ax.set_title('效率 vs 效能', fontproperties=fprop(11), color='#e0e0ff')
ax.grid(True)
ax.legend(handles=[mpatches.Patch(color=c, label=t)
                   for t, c in [('傳統 ML', '#7bb4f7'), ('深度學習', '#7bf7c8')]],
          prop=fprop(9))

ax2 = axes2[1]
rec_data  = df_plot['Recall_macro'].sort_values(ascending=True)
ax2.barh(range(len(rec_data)), rec_data,
         color=[model_color(n) for n in rec_data.index], edgecolor='#0d0d1a')
ax2.set_yticks(range(len(rec_data)))
ax2.set_yticklabels(rec_data.index, fontproperties=fprop(9))
for i, v in enumerate(rec_data):
    ax2.text(v + 0.003, i, f'{v:.4f}', va='center',
             fontproperties=fprop(8.5), color='#c0c8ff')
ax2.set_xlabel('Recall (macro)', fontproperties=fprop(10))
ax2.set_title('各故障類別召回率（macro）', fontproperties=fprop(11), color='#e0e0ff')
ax2.grid(True, axis='x')
ax2.set_xlim(0, 1.1)

plt.tight_layout()
plt.savefig(SAVE_DIR + 'fig2_efficiency.png', dpi=150,
            bbox_inches='tight', facecolor='#0d0d1a')
plt.close()
print("✓ 圖2")

# ════════════════════════════════════════════════════════════════════════
# 圖3：混淆矩陣（Stage 2，4 類）
# ════════════════════════════════════════════════════════════════════════
fig3, axes3 = plt.subplots(1, 2, figsize=(14, 6))
fig3.patch.set_facecolor('#0d0d1a')
fig3.suptitle('混淆矩陣比較：LightGBM vs MLP (upgraded)（Stage 2）',
              fontproperties=fprop(13), color='#e0e8ff')

for ax, (mname, color) in zip(axes3,
        [('LightGBM', '#7bb4f7'), ('MLP (upgraded)', '#7bf7c8')]):
    yp = models2[mname].predict(Xte2)
    cm = confusion_matrix(yte2, yp, labels=[0, 1, 2, 3])
    im = ax.imshow(cm, cmap='Blues', aspect='auto')
    ax.set_xticks(range(4)); ax.set_yticks(range(4))
    ax.set_xticklabels(CLS_LABELS_S2, fontproperties=fprop(9))
    ax.set_yticklabels(CLS_LABELS_S2, fontproperties=fprop(9))
    for i in range(4):
        for j in range(4):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                    fontproperties=fprop(9),
                    color='white' if cm[i, j] > cm.max() * 0.5 else '#c0c8ff')
    ax.set_xlabel('預測', fontproperties=fprop(10))
    ax.set_ylabel('實際', fontproperties=fprop(10))
    ax.set_title(f'{mname}  (MCC={df_r.loc[mname, "MCC"]:.4f})',
                 fontproperties=fprop(11), color=color)
    plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.savefig(SAVE_DIR + 'fig3_confusion.png', dpi=150,
            bbox_inches='tight', facecolor='#0d0d1a')
plt.close()
print("✓ 圖3")

# ════════════════════════════════════════════════════════════════════════
# 圖4：SHAP + XGBoost vs TabNet 特徵重要性對比
# ════════════════════════════════════════════════════════════════════════
fig4, axes4 = plt.subplots(1, 2, figsize=(16, 7))
fig4.patch.set_facecolor('#0d0d1a')
fig4.suptitle('可解釋性分析：XGBoost SHAP vs TabNet Sparse Attention',
              fontproperties=fprop(13), color='#e0e8ff')

ax = axes4[0]
y_pos = 0
yticks_pos, yticks_lbl = [], []
for cls in [3, 2, 1]:   # OSF → PWF → HDF
    fname_zh = FAILURE_MAP_S2[cls]
    top3     = shap_by_class.get(cls, [])
    color    = FAILURE_COLOR[fname_zh]
    for ri, (fn, sv) in enumerate(top3):
        ax.barh(y_pos, sv, 0.7, color=color,
                alpha=0.9 - ri * 0.22, edgecolor='#0d0d1a')
        ax.text(sv + 0.05, y_pos, f'{fn} ({sv:.2f})',
                va='center', fontproperties=fprop(8.5), color='#d0d0ee')
        if fn == PHYSICS_S2[cls] and ri == 0:
            ax.text(-0.15, y_pos, '✓', va='center',
                    fontproperties=fprop(11), color='#7bf7c8')
        y_pos += 0.85
    yticks_pos.append(y_pos - 0.85 * 1.5)
    yticks_lbl.append(fname_zh)
    y_pos += 0.5

ax.set_xlim(-0.5, 8)
ax.set_yticks(yticks_pos)
ax.set_yticklabels(yticks_lbl, fontproperties=fprop(9))
ax.set_xlabel('mean |SHAP value|', fontproperties=fprop(10))
ax.set_title('XGBoost SHAP（各故障類別）\n✓ = 符合物理觸發條件',
             fontproperties=fprop(10), color='#e0e0ff')
ax.grid(True, axis='x')

ax2 = axes4[1]
fn_order  = sorted(xgb_fi, key=lambda k: xgb_fi[k] + tabnet_fi.get(k, 0))
x = np.arange(len(fn_order)); w = 0.35
ax2.barh(x - w/2, [xgb_fi.get(f, 0) for f in fn_order],
         w, label='XGBoost', color='#f7c47b', alpha=0.85, edgecolor='#0d0d1a')
ax2.barh(x + w/2, [tabnet_fi.get(f, 0) for f in fn_order],
         w, label='TabNet',  color='#7bf7c8', alpha=0.85, edgecolor='#0d0d1a')
ax2.set_yticks(x)
ax2.set_yticklabels(fn_order, fontproperties=fprop(9))
ax2.set_xlabel('Feature Importance', fontproperties=fprop(10))
ax2.set_title('全局特徵重要性對比\nXGBoost vs TabNet',
              fontproperties=fprop(10), color='#e0e0ff')
ax2.legend(prop=fprop(9))
ax2.grid(True, axis='x')

plt.tight_layout()
plt.savefig(SAVE_DIR + 'fig4_shap_tabnet.png', dpi=150,
            bbox_inches='tight', facecolor='#0d0d1a')
plt.close()
print("✓ 圖4")

# ════════════════════════════════════════════════════════════════════════
# 圖5：研究架構與核心結論摘要
# ════════════════════════════════════════════════════════════════════════
fig5 = plt.figure(figsize=(18, 8))
fig5.patch.set_facecolor('#0d0d1a')
gs = GridSpec(1, 3, figure=fig5, wspace=0.3)

# 左：EDA 可診斷性（用實驗A實際數字）
ax_l = fig5.add_subplot(gs[0, 0])
ax_l.set_xlim(0, 10); ax_l.set_ylim(0, 10); ax_l.axis('off')
ax_l.add_patch(plt.Rectangle((0,0), 10, 10, linewidth=1,
               edgecolor='#2a2a4a', facecolor='#0f0f1a'))
ax_l.text(5, 9.3, 'EDA 階段：故障可診斷性', ha='center',
          fontproperties=fprop(11), color='#7bf7c8')
for i, (name, nature, auc_s, c) in enumerate([
    ('HDF 散熱不良', '規則型',  'AUC=1.0000 [OK]', '#f97316'),
    ('PWF 功率異常', '規則型',  'AUC=0.9992 [OK]', '#fbbf24'),
    ('OSF 過負荷',  '規則型',  'AUC=0.9998 [OK]', '#a78bfa'),
    ('TWF 刀具磨耗', '半隨機型', 'AUC=0.9637 [!!]', '#f87171'),
    ('RNF 隨機故障', '純隨機型', 'AUC=0.6629 [X]', '#888888'),
]):
    y = 8.0 - i * 1.5
    ax_l.add_patch(plt.Rectangle((0.3, y-0.45), 9.4, 1.1,
                   linewidth=0.8, edgecolor=c, facecolor=c+'22', alpha=0.7))
    ax_l.text(0.7, y+0.15, name,   fontproperties=fprop(9),   color=c,        va='center')
    ax_l.text(0.7, y-0.2,  nature, fontproperties=fprop(8),   color='#8888aa', va='center')
    ax_l.text(9.3, y,      auc_s,  fontproperties=fprop(8.5), color=c,        va='center', ha='right')

# 中：模型排名
ax_m = fig5.add_subplot(gs[0, 1])
ax_m.set_xlim(0, 10); ax_m.set_ylim(0, 10); ax_m.axis('off')
ax_m.add_patch(plt.Rectangle((0,0), 10, 10, linewidth=1,
               edgecolor='#2a2a4a', facecolor='#0f0f1a'))
ax_m.text(5, 9.3, '模型效能排名（MCC）', ha='center',
          fontproperties=fprop(11), color='#7bb4f7')
for i, (name, mcc) in enumerate(df_r['MCC'].sort_values(ascending=False).items()):
    y      = 8.2 - i * 0.87
    medal  = {0:'🥇', 1:'🥈', 2:'🥉'}.get(i, '')
    tag    = 'DL' if name in DL_MODELS else 'ML'
    tag_c  = '#7bf7c8' if tag == 'DL' else '#7bb4f7'
    ax_m.add_patch(plt.Rectangle((1.5, y-0.28), mcc*6, 0.56,
                   linewidth=0, facecolor=model_color(name), alpha=0.7))
    ax_m.text(0.2,          y, f'{medal}{i+1}.', fontproperties=fprop(9),   va='center', color='#c0c8ff')
    ax_m.text(1.4,          y, name,             fontproperties=fprop(8.5), va='center', color=model_color(name), ha='right')
    ax_m.text(1.5+mcc*6+0.1, y, f'{mcc:.4f}',   fontproperties=fprop(8.5), va='center', color='#c0c8ff')
    ax_m.text(9.5,          y, f'[{tag}]',       fontproperties=fprop(7.5), va='center', color=tag_c, ha='right')

# 右：核心結論（用實際數字）
ax_r = fig5.add_subplot(gs[0, 2])
ax_r.set_xlim(0, 10); ax_r.set_ylim(0, 10); ax_r.axis('off')
ax_r.add_patch(plt.Rectangle((0,0), 10, 10, linewidth=1,
               edgecolor='#2a2a4a', facecolor='#0f0f1a'))
ax_r.text(5, 9.3, '核心研究結論', ha='center',
          fontproperties=fprop(11), color='#c47bf7')
for y_c, (tag, title, text, c) in zip([8.3, 6.3, 4.3, 2.3], [
    ('C1', '故障本質決定預測天花板',
     '規則型 AUC>0.999，半隨機型 AUC=0.964\n純隨機型無法預測→EDA 階段排除', '#7bf7c8'),
    ('C2', '排除雜訊後效能顯著提升',
     '兩階段架構使各模型 MCC 平均提升 +0.16\nML 樹狀模型在低維規則邊界優於 DL', '#7bb4f7'),
    ('C3', '衍生特徵有實質貢獻',
     '功率為最重要特徵（ΔMCC=−0.019）\n完整特徵 MCC=0.808 vs 僅原始=0.719', '#f7c47b'),
    ('C4', '模型學到領域知識',
     'SHAP Top1 與物理規則 3/3 一致\nTabNet 特徵重要性排序亦相同', '#c47bf7'),
]):
    ax_r.add_patch(plt.Rectangle((0.2, y_c-0.75), 9.6, 1.65,
                   linewidth=0.8, edgecolor=c, facecolor=c+'18', alpha=0.8))
    ax_r.text(0.6, y_c+0.45, f'[{tag}] {title}', fontproperties=fprop(8.5), color=c, va='center')
    # 右欄：結論文字改為較亮的顏色
    for li, line in enumerate(text.split('\n')):
        ax_r.text(0.6, y_c+0.05-li*0.38, line,
              fontproperties=fprop(7.8), color='#e0e0ff',  # ← '#b0b0cc' 改為 '#e0e0ff'
              va='center')

fig5.suptitle('AI4I 2020 預測性維護研究 — 完整研究架構與核心結論',
              fontproperties=fprop(13), color='#e0e8ff', y=1.01)
plt.savefig(SAVE_DIR + 'fig5_summary.png', dpi=150,
            bbox_inches='tight', facecolor='#0d0d1a')
plt.close()
print("✓ 圖5")

# 最終 CSV
df_r.sort_values('MCC', ascending=False).to_csv(
    SAVE_DIR + 'final_results.csv', encoding='utf-8-sig')
print("✓ final_results.csv 已儲存")
print("\n最終成績單：")
print(df_r[['MCC', 'F1w', 'AUC', 'Recall_macro']].sort_values('MCC', ascending=False).round(4).to_string())

✓ 圖1
✓ 圖2
✓ 圖3
✓ 圖4
✓ 圖5
✓ final_results.csv 已儲存

最終成績單：
                           MCC     F1w     AUC  Recall_macro
LightGBM                0.9291  0.9960  0.9997        0.9450
Gradient Boosting       0.9247  0.9957  0.9995        0.9725
Stacking (XGB+LGBM→RF)  0.9163  0.9953  0.9761        0.8794
XGBoost                 0.9106  0.9951  0.9998        0.9184
Decision Tree           0.8982  0.9944  0.9260        0.8832
Random Forest           0.8910  0.9940  0.9992        0.8949
MLP (upgraded)          0.7958  0.9871  0.9977        0.9381
TabNet                  0.7842  0.9873  0.9980        0.8729
MLP (original)          0.7500  0.9840  0.9968        0.9108
KNN                     0.5361  0.9695  0.8584        0.7238


## 9：深度分析圖(三階段對比 + 貝葉斯誤差說明)

In [16]:
# ── Cell 9：深度分析圖（三階段對比 + 貝葉斯誤差說明）────────────────
# 依賴 results1、results2（含 TabNet）

from matplotlib.gridspec import GridSpec

fig = plt.figure(figsize=(18, 10))
fig.patch.set_facecolor('#0d0d1a')
gs  = GridSpec(1, 2, figure=fig, wspace=0.35)

# ════════════════════════════════════════════════════════════════════════
# 左：三階段 MCC 對比（Stage 1 vs Stage 2）
# ════════════════════════════════════════════════════════════════════════
ax1 = fig.add_subplot(gs[0, 0])

common = ['KNN', 'Decision Tree', 'Random Forest', 'Gradient Boosting',
          'XGBoost', 'LightGBM', 'MLP (original)', 'MLP (upgraded)',
          'Stacking (XGB+LGBM→RF)']

s1_mcc = [results1[m]['MCC'] for m in common]
s2_mcc = [results2[m]['MCC'] for m in common]

x = np.arange(len(common))
w = 0.35
ax1.barh(x - w/2, s1_mcc, w, label='Stage 1（排除 RNF）',
         color='#7bb4f7', alpha=0.75, edgecolor='#0d0d1a')
ax1.barh(x + w/2, s2_mcc, w, label='Stage 2（排除 RNF + TWF）',
         color='#7bf7c8', alpha=0.85, edgecolor='#0d0d1a')

# 標示提升幅度
for i, (s1, s2) in enumerate(zip(s1_mcc, s2_mcc)):
    delta = s2 - s1
    ax1.text(max(s1, s2) + 0.005, i,
             f'+{delta:.3f}', va='center',
             fontproperties=fprop(8), color='#f7c47b')

ax1.set_yticks(x)
ax1.set_yticklabels(common, fontproperties=fprop(9))
ax1.set_xlabel('MCC', fontproperties=fprop(10))
ax1.set_xlim(0.4, 1.05)
ax1.set_title('兩階段雜訊排除的效能提升\nStage 1 vs Stage 2 MCC 對比',
              fontproperties=fprop(11), color='#e0e0ff')
ax1.legend(prop=fprop(9))
ax1.grid(True, axis='x')

# 平均提升標示
avg_delta = np.mean([s2 - s1 for s1, s2 in zip(s1_mcc, s2_mcc)])
ax1.text(0.97, -0.8, f'平均提升 +{avg_delta:.3f}',
         ha='right', fontproperties=fprop(9), color='#f7c47b',
         transform=ax1.get_yaxis_transform())

# ════════════════════════════════════════════════════════════════════════
# 右：貝葉斯誤差天花板說明
# ════════════════════════════════════════════════════════════════════════
ax2 = fig.add_subplot(gs[0, 1])
ax2.set_xlim(0, 10); ax2.set_ylim(0, 10); ax2.axis('off')
ax2.add_patch(plt.Rectangle((0, 0), 10, 10, linewidth=1,
              edgecolor='#2a2a4a', facecolor='#0f0f1a'))
ax2.text(5, 9.4, '故障可預測性天花板分析', ha='center',
         fontproperties=fprop(12), color='#e0e0ff')

segments = [
    ('#7bf7c8', 'HDF / PWF / OSF（規則型）', [
        '物理觸發條件明確（溫差、功率、功率×磨耗）',
        'AUC > 0.999，已接近理論極限',
        '→ Stage 2 主要學習目標',
    ]),
    ('#f7c47b', 'TWF（半隨機型）', [
        'Tool wear 200 min 後進入隨機觸發區',
        '各區間發生率 4-16%，無單調趨勢',
        '→ 條件隨機，Stage 2 排除',
    ]),
    ('#f87171', 'RNF（純隨機型）', [
        '與所有製程參數無關，純隨機 0.1%',
        'AUC=0.6629，Recall=0（模型無法抓到）',
        '→ 純隨機，Stage 1 起即排除',
    ]),
]

y_cur = 8.2
for c, title, items in segments:
    ax2.add_patch(plt.Rectangle((0.3, y_cur - 1.1), 9.4, 2.0,
                 linewidth=0.8, edgecolor=c, facecolor=c+'20', alpha=0.8))
    ax2.text(0.7, y_cur + 0.5, title,
             fontproperties=fprop(9.5), color=c, va='center')
    for li, line in enumerate(items):
        ax2.text(0.7, y_cur + 0.05 - li * 0.42, line,
                 fontproperties=fprop(8), color='#e0e0ff', va='center')
    y_cur -= 3.0

fig.suptitle('兩階段預測性維護架構 — 雜訊排除效果與可預測性天花板',
             fontproperties=fprop(13), color='#e0e8ff', y=1.02)
plt.savefig(SAVE_DIR + 'fig_deep_analysis.png', dpi=150,
            bbox_inches='tight', facecolor='#0d0d1a')
plt.close()
print("✓ fig_deep_analysis.png 已儲存")

✓ fig_deep_analysis.png 已儲存


## 10. 圖表：三實驗完整報告圖

整合實驗A（可診斷性）、實驗B（消融實驗）、實驗C（SHAP）的視覺化報告。

In [17]:
# ── Cell 10：三實驗完整報告圖（ABC）─────────────────────────────────
# 依賴 Cell 2 的 df_a、Cell 3 的 df_b、Cell 6 的 shap_by_class
# 所有實驗已跑完，此 Cell 只負責整合視覺化

from matplotlib.gridspec import GridSpec

# 讀取已儲存的 CSV（確保數字與 Cell 2/3/6 一致）
df_a = pd.read_csv(SAVE_DIR + 'expA_diagnostics.csv')
df_b = pd.read_csv(SAVE_DIR + 'expB_ablation.csv')

# shap_results 從 results_stage2.pkl 取回
with open(SAVE_DIR + 'results_stage2.pkl', 'rb') as f:
    s2 = pickle.load(f)
shap_by_class = s2['shap_by_class']  # key: 1=HDF, 2=PWF, 3=OSF

FAILURE_COLS   = ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']
FAILURE_NATURE = {'TWF':'半隨機型','HDF':'規則型','PWF':'規則型',
                  'OSF':'規則型','RNF':'純隨機型'}
NATURE_COLOR   = {'規則型':'#7bf7c8','半隨機型':'#f7c47b','純隨機型':'#f87171'}
FAILURE_ZH_1L  = {'TWF':'刀具磨耗 (TWF)','HDF':'散熱不良 (HDF)',
                  'PWF':'功率異常 (PWF)','OSF':'過負荷 (OSF)','RNF':'隨機故障 (RNF)'}
FAILURE_ZH_2L  = {'TWF':'刀具磨耗\n(TWF)','HDF':'散熱不良\n(HDF)',
                  'PWF':'功率異常\n(PWF)','OSF':'過負荷\n(OSF)','RNF':'隨機故障\n(RNF)'}

# Stage 2 SHAP 對應（key 1=HDF, 2=PWF, 3=OSF）
SHAP_FC_MAP  = {1:'HDF', 2:'PWF', 3:'OSF'}
PHYSICS_TOP1 = {'HDF':'溫差', 'PWF':'功率', 'OSF':'功率×磨耗'}

colors_a = [NATURE_COLOR[FAILURE_NATURE[fc]] for fc in FAILURE_COLS]

fig = plt.figure(figsize=(20, 18))
fig.patch.set_facecolor('#0d0d1a')
gs  = GridSpec(3, 3, figure=fig, hspace=0.48, wspace=0.35)

# ── 圖1：各故障 AUC ──────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, :2])
fc_labels_2l = [FAILURE_ZH_2L[fc] for fc in FAILURE_COLS]
bars = ax1.bar(fc_labels_2l, df_a['AUC'], color=colors_a,
               edgecolor='#0d0d1a', linewidth=1.5, width=0.55)
for bar, val in zip(bars, df_a['AUC']):
    c_txt = '#f87171' if val < 0.7 else '#7bf7c8' if val > 0.99 else '#f7c47b'
    ax1.text(bar.get_x() + bar.get_width()/2, val + 0.005,
             f'{val:.4f}', ha='center', va='bottom',
             fontproperties=fprop(10), color=c_txt)
ax1.set_ylim(0, 1.12)
ax1.set_ylabel('ROC-AUC', fontproperties=fprop(10))
ax1.set_title('【實驗A】各故障類型可診斷性 — ROC-AUC',
              fontproperties=fprop(12), color='#e0e0ff')
ax1.axhline(0.90, color='#8888aa', linestyle='--', alpha=0.5, linewidth=1)
ax1.text(4.3, 0.91, 'AUC=0.90', color='#8888aa',
         fontproperties=fprop(8))
ax1.grid(True, axis='y')
ax1.legend(handles=[mpatches.Patch(color=c, label=n)
                    for n, c in NATURE_COLOR.items()],
           loc='lower right', prop=fprop(9))
for label in ax1.get_xticklabels():
    label.set_fontproperties(fprop(9))

# ── 圖2：各故障 Recall ───────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 2])
fc_labels_1l = [FAILURE_ZH_1L[fc] for fc in FAILURE_COLS]
bars2 = ax2.barh(fc_labels_1l[::-1], df_a['Recall'].values[::-1],
                 color=colors_a[::-1], edgecolor='#0d0d1a', height=0.5)
for bar, val in zip(bars2, df_a['Recall'].values[::-1]):
    ax2.text(val + 0.01, bar.get_y() + bar.get_height()/2,
             f'{val:.3f}', va='center', fontproperties=fprop(9))
ax2.set_xlim(0, 1.18)
ax2.set_xlabel('Recall', fontproperties=fprop(10))
ax2.set_title('各故障 Recall', fontproperties=fprop(11), color='#e0e0ff')
ax2.axvline(0.5, color='#8888aa', linestyle='--', alpha=0.4)
ax2.grid(True, axis='x')
for label in ax2.get_yticklabels():
    label.set_fontproperties(fprop(8))

# ── 圖3：消融實驗 ────────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, :2])
remove_df  = df_b[df_b['特徵組合'].str.startswith('移除')].copy()
baseline_v = df_b.loc[df_b['特徵組合'] == '完整特徵（基準）', 'MCC'].values[0]
labels_r   = remove_df['特徵組合'].str.replace('移除 ', '', regex=False).values
mccs_r     = remove_df['MCC'].values
delta_r    = remove_df['ΔMCC'].values
bar_colors3 = ['#f87171' if d < -0.02 else '#f7c47b' if d < -0.005
               else '#7bb4f7' for d in delta_r]

ax3.axhline(baseline_v, color='#7bf7c8', linestyle='--', alpha=0.7,
            linewidth=1.5, label=f'基準 MCC={baseline_v:.4f}')
bars3 = ax3.bar(range(len(labels_r)), mccs_r, color=bar_colors3,
                edgecolor='#0d0d1a', linewidth=0.8, width=0.6)
for bar, d in zip(bars3, delta_r):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
             f'{d:+.3f}', ha='center',
             fontproperties=fprop(7.5),
             color='#f87171' if d < -0.015 else '#8888aa')

clean_labels = [l.replace(' [K]','').replace(' [rpm]','')
                 .replace(' [min]','').replace(' [Nm]','')
                for l in labels_r]
ax3.set_xticks(range(len(clean_labels)))
ax3.set_xticklabels(clean_labels, rotation=30, ha='right')
for label in ax3.get_xticklabels():
    label.set_fontproperties(fprop(8.5))
ax3.set_ylabel('MCC', fontproperties=fprop(10))
ax3.set_ylim(0.65, 0.88)
ax3.set_title('【實驗B】消融實驗 — 移除各特徵後的 MCC 變化',
              fontproperties=fprop(12), color='#e0e0ff')
ax3.legend(prop=fprop(9))
ax3.grid(True, axis='y')
ax3.axvspan(-0.5, 3.5, alpha=0.06, color='#7bf7c8')
ax3.axvspan(3.5, len(labels_r) - 0.5, alpha=0.06, color='#7bb4f7')
ax3.text(1.5, 0.865, '衍生特徵', ha='center',
         color='#7bf7c8', fontproperties=fprop(8))
ax3.text(7.0, 0.865, '原始特徵', ha='center',
         color='#7bb4f7', fontproperties=fprop(8))

# ── 圖4：三組特徵組合比較 ─────────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 2])
grp = df_b[df_b['特徵組合'].isin(['完整特徵（基準）', '只用原始特徵', '只用衍生特徵'])]
labels_g = ['完整\n(11個)', '僅原始\n(7個)', '僅衍生\n(4個)']
for metric, offset, color in [('MCC', -0.12, '#7bf7c8'), ('AUC', 0.12, '#7bb4f7')]:
    vals = grp[metric].values
    x    = np.array([0, 1, 2]) + offset
    ax4.bar(x, vals, 0.2, color=color, alpha=0.8,
            label=metric, edgecolor='#0d0d1a')
    for xi, v in zip(x, vals):
        ax4.text(xi, v + 0.005, f'{v:.3f}', ha='center',
                 fontproperties=fprop(8))
ax4.set_xticks([0, 1, 2])
ax4.set_xticklabels(labels_g)
for label in ax4.get_xticklabels():
    label.set_fontproperties(fprop(9))
ax4.set_ylim(0.55, 1.05)
ax4.set_title('特徵組合對比', fontproperties=fprop(11), color='#e0e0ff')
ax4.legend(prop=fprop(9))
ax4.grid(True, axis='y')

# ── 圖5：SHAP Top 特徵（Stage 2：HDF/PWF/OSF）───────────────────────
ax5 = fig.add_subplot(gs[2, :2])
fc_order_s2 = [1, 2, 3]   # HDF, PWF, OSF
y_pos = np.arange(len(fc_order_s2)) * 3.5
FC_COLOR = {'HDF':'#f97316', 'PWF':'#fbbf24', 'OSF':'#a78bfa'}

for fi, cls in enumerate(fc_order_s2):
    fc      = SHAP_FC_MAP[cls]
    items   = shap_by_class.get(cls, [])
    y_base  = y_pos[fi]
    nc      = FC_COLOR[fc]
    for ri, (fname, val) in enumerate(items[:3]):
        ax5.barh(y_base + ri * 0.75, val, 0.7, color=nc,
                 alpha=0.9 - ri * 0.25, edgecolor='#0d0d1a')
        ax5.text(val + 0.05, y_base + ri * 0.75,
                 f'{fname} ({val:.2f})', va='center',
                 fontproperties=fprop(8.5), color='#c0c8ff')
        if fname == PHYSICS_TOP1[fc] and ri == 0:
            ax5.text(-0.2, y_base + ri * 0.75, '✓',
                     va='center', fontproperties=fprop(11), color='#7bf7c8')
    ax5.text(-0.4, y_base + 0.75,
             f'{fc}\n({FAILURE_NATURE[fc]})', ha='right', va='center',
             fontproperties=fprop(9), color=nc)

ax5.set_yticks([])
ax5.set_xlabel('mean |SHAP value|', fontproperties=fprop(10))
ax5.set_xlim(-1.5, 8)
ax5.set_title('【實驗C】SHAP 可解釋性 — Stage 2 各故障主要決策特徵\n'
              '（✓ = 符合物理觸發條件，3/3 一致）',
              fontproperties=fprop(12), color='#e0e0ff')
ax5.grid(True, axis='x')

# ── 圖6：三實驗核心結論 ──────────────────────────────────────────────
ax6 = fig.add_subplot(gs[2, 2])
ax6.set_xlim(0, 10); ax6.set_ylim(0, 10); ax6.axis('off')
ax6.add_patch(plt.Rectangle((0, 0), 10, 10, linewidth=1,
              edgecolor='#2a2a4a', facecolor='#0f0f1a'))
ax6.set_title('三實驗核心結論', fontproperties=fprop(11), color='#e0e0ff')

conclusions = [
    ('實驗A：可診斷性', '#7bf7c8', [
        'HDF/PWF/OSF（規則型）：AUC > 0.999',
        'TWF（半隨機）：AUC=0.9637',
        'RNF（純隨機）：AUC=0.6629，Recall=0',
        '→ 故障本質決定預測天花板',
    ]),
    ('實驗B：特徵工程', '#f7c47b', [
        '移除「功率」：ΔMCC=-0.019（最重要）',
        '完整特徵 MCC=0.808 vs 僅原始=0.719',
        '→ 衍生特徵有實質貢獻',
    ]),
    ('實驗C：可解釋性', '#c47bf7', [
        'SHAP Top1 與物理規則 3/3 一致',
        'TabNet 特徵重要性排序亦相同',
        '→ 模型學到領域知識',
    ]),
]
y_cur = 9.5
for title, color, items in conclusions:
    ax6.text(0.2, y_cur, title, color=color,
             fontproperties=fprop(9.5))
    y_cur -= 0.65
    for item in items:
        ax6.text(0.5, y_cur, item, color='#e0e0ff',
                 fontproperties=fprop(7.8))
        y_cur -= 0.58
    y_cur -= 0.3

plt.suptitle('AI4I 2020 預測性維護 — 故障可診斷性、特徵工程貢獻度、可解釋性 三項深度實驗',
             fontproperties=fprop(13), color='#e0e8ff', y=1.01)
plt.savefig(SAVE_DIR + 'experiments_ABC_report.png', dpi=150,
            bbox_inches='tight', facecolor='#0d0d1a')
plt.close()
print("✓ experiments_ABC_report.png 已儲存 →", SAVE_DIR)

✓ experiments_ABC_report.png 已儲存 → /content/drive/MyDrive/115專題/ai4i_complete_code/


## 11. 圖表：兩階段架構總覽圖

In [18]:
# ── Cell 11：兩階段決策架構流程圖────────────────────────────────────

fig, ax = plt.subplots(figsize=(14, 16))
fig.patch.set_facecolor('#0d0d1a')
ax.set_xlim(0, 14); ax.set_ylim(0, 16)
ax.axis('off')
ax.set_facecolor('#0d0d1a')

def box(ax, x, y, w, h, text, color, fontsize=9, text_color='#e0e0ff'):
    ax.add_patch(plt.Rectangle((x - w/2, y - h/2), w, h,
                 linewidth=1.5, edgecolor=color,
                 facecolor=color + '25', zorder=3))
    lines = text.split('\n')
    total = len(lines)
    for i, line in enumerate(lines):
        offset = ((total - 1) / 2 - i) * 0.22
        ax.text(x, y + offset, line, ha='center', va='center',
                fontproperties=fprop(fontsize), color=text_color, zorder=4)

def arrow(ax, x1, y1, x2, y2, color='#555577'):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color=color, lw=1.8), zorder=2)

def side_note(ax, x, y, text, color):
    ax.text(x, y, text, ha='left', va='center',
            fontproperties=fprop(8), color=color,
            bbox=dict(boxstyle='round,pad=0.3', facecolor=color+'15',
                      edgecolor=color, linewidth=0.8))

# ── 主流程（垂直置中）────────────────────────────────────────────────
CX = 7  # 中心 X

# 1. 資料輸入
box(ax, CX, 15.2, 6, 1.0,
    'AI4I 2020 資料集\n10,000 筆 × 14 欄位\n5 種故障類型（TWF / HDF / PWF / OSF / RNF）',
    '#7bb4f7', fontsize=9)

# 2. EDA 可診斷性分析
arrow(ax, CX, 14.7, CX, 13.85)
box(ax, CX, 13.4, 7, 0.85,
    'EDA：故障可診斷性分析（實驗A）\n1-vs-Rest XGBoost 5-Fold CV',
    '#aaaaaa', fontsize=9)
side_note(ax, 10.8, 13.4, '實驗A', '#7bf7c8')

# 3. 可診斷性結果（三欄）
arrow(ax, CX, 12.97, CX, 12.4)
for xp, label_text, auc_text, c in [
    (3.0,  'HDF / PWF / OSF\n規則型',   'AUC > 0.999 ✓', '#7bf7c8'),
    (7.0,  'TWF  半隨機型',             'AUC = 0.9637',   '#f7c47b'),
    (11.0, 'RNF  純隨機型',             'AUC = 0.6629',   '#f87171'),
]:
    box(ax, xp, 11.85, 3.5, 0.95,
        f'{label_text}\n{auc_text}', c, fontsize=8.5)

# 從 EDA 框分叉到三欄
for xp in [3.0, 7.0, 11.0]:
    ax.annotate('', xy=(xp, 12.33), xytext=(CX, 12.97),
                arrowprops=dict(arrowstyle='->', color='#555577', lw=1.3))

# 4. 排除 RNF（主線決策）
arrow(ax, CX, 11.37, CX, 10.75)
box(ax, CX, 10.45, 6.5, 0.55,
    '排除 RNF（純隨機，無特徵可預測）→ 剩 9,981 筆',
    '#f87171', fontsize=8.5, text_color='#f87171')

# 5. Stage 1
arrow(ax, CX, 10.17, CX, 9.45)
box(ax, CX, 9.0, 8, 0.85,
    'Stage 1：5 類分類（正常 / TWF / HDF / PWF / OSF）\n9 個模型  |  最佳：Stacking MCC = 0.8716',
    '#7bb4f7', fontsize=9)
side_note(ax, 11.3, 9.0, '實驗B', '#f7c47b')

# 6. 排除 TWF（主線決策）
arrow(ax, CX, 8.57, CX, 7.95)
box(ax, CX, 7.65, 6.5, 0.55,
    '排除 TWF（條件隨機，各區間發生率不穩定）→ 剩 9,936 筆',
    '#f7c47b', fontsize=8.5, text_color='#f7c47b')

# 7. Stage 2
arrow(ax, CX, 7.37, CX, 6.65)
box(ax, CX, 6.2, 8, 0.85,
    'Stage 2：4 類分類（正常 / HDF / PWF / OSF）\n10 個模型（含 TabNet）  |  最佳：LightGBM MCC = 0.9291',
    '#7bf7c8', fontsize=9)
side_note(ax, 11.3, 6.2, '實驗C', '#c47bf7')

# 8. 效能提升標示
arrow(ax, CX, 5.77, CX, 5.15)
box(ax, CX, 4.85, 6.5, 0.55,
    '兩階段排除效果：各模型 MCC 平均提升 +0.16',
    '#aaaaaa', fontsize=8.5, text_color='#aaaaaa')

# 9. 輸出
arrow(ax, CX, 4.57, CX, 3.95)
box(ax, CX, 3.55, 7, 0.75,
    '輸出：故障診斷結果\nSHAP 可解釋性報告（3/3 符合物理規則）',
    '#c47bf7', fontsize=9)

# 標題
ax.set_title('兩階段預測性維護決策支援系統 — 研究架構流程圖',
             fontproperties=fprop(13), color='#e0e8ff', pad=15)

plt.tight_layout()
plt.savefig(SAVE_DIR + 'fig_two_stage.png', dpi=150,
            bbox_inches='tight', facecolor='#0d0d1a')
plt.close()
print("✓ fig_two_stage.png 已儲存 →", SAVE_DIR)

✓ fig_two_stage.png 已儲存 → /content/drive/MyDrive/115專題/ai4i_complete_code/


# Streamlit UI

In [19]:
# ── Cell 12：產生 app_bundle.pkl（供 Streamlit 使用）────────────────
# 依賴 Cell 4b 的 results1、models1
#       Cell 5b 的 results2、models2、stage2_state
#       Cell 6  的 shap_by_class、xgb_fi
#       Cell 7  的 results2['TabNet']、tn_fi

import pickle

# Stage 2 測試集
X_test  = stage2_state['X_test_sc'].values
y_test  = np.array(stage2_state['y_test']).astype(int)
X_train = stage2_state['X_train_sc'].values
y_train = np.array(stage2_state['y_train']).astype(int)

# 從 pkl 取回 SHAP 資料
with open(SAVE_DIR + 'results_stage2.pkl', 'rb') as f:
    s2 = pickle.load(f)

# results2 轉 DataFrame（供 results.csv 和 bundle 使用）
df_results2 = pd.DataFrame(results2).T
df_results2.index.name = 'model'

# results1 轉 DataFrame（供三階段對比）
df_results1 = pd.DataFrame(results1).T
df_results1.index.name = 'model'

# 三階段 MCC 對比表
common_models = [m for m in results1 if m in results2]
three_stage = {}
for m in common_models:
    three_stage[m] = {
        '排除RNF':     round(results1[m]['MCC'], 4),
        '排除TWF_RNF': round(results2[m]['MCC'], 4),
    }
if 'TabNet' in results2:
    three_stage['TabNet'] = {
        '排除RNF':     None,
        '排除TWF_RNF': round(results2['TabNet']['MCC'], 4),
    }

bundle = dict(
    # 模型與資料
    models      = models2,
    scaler      = stage2_state['scaler'],
    feat_cols   = stage2_state['feat_cols'],   # FEAT_COLS_SAFE
    X_test      = X_test,
    y_test      = y_test,
    X_train     = X_train,
    y_train     = y_train,

    # 結果
    results     = df_results2,
    stage1_results = df_results1,
    three_stage = three_stage,

    # SHAP
    shap_by_class = s2['shap_by_class'],
    xgb_fi        = s2['xgb_fi'],
    tabnet_fi     = s2['tabnet_fi'],
    feat_short    = s2['feat_short'],
)

# 儲存到 Drive
bundle_path = SAVE_DIR + 'app_bundle.pkl'
with open(bundle_path, 'wb') as f:
    pickle.dump(bundle, f)

print("✓ app_bundle.pkl 已儲存 →", bundle_path)
print(f"\n內容確認：")
print(f"  模型數：{len(models2)} 個 → {list(models2.keys())}")
print(f"  特徵數：{len(stage2_state['feat_cols'])} 個")
print(f"  測試集：{X_test.shape}")
print(f"  訓練集：{X_train.shape}")
print(f"  Stage 2 結果：{df_results2.shape}")
print(f"  三階段對比：{len(three_stage)} 個模型")
print(f"\n三階段 MCC：")
for m, v in three_stage.items():
    s1 = f"{v['排除RNF']:.4f}" if v['排除RNF'] else '  —  '
    s2_v = f"{v['排除TWF_RNF']:.4f}" if v['排除TWF_RNF'] else '  —  '
    print(f"  {m:30s} Stage1={s1}  Stage2={s2_v}")

✓ app_bundle.pkl 已儲存 → /content/drive/MyDrive/115專題/ai4i_complete_code/app_bundle.pkl

內容確認：
  模型數：10 個 → ['KNN', 'Decision Tree', 'Random Forest', 'Gradient Boosting', 'XGBoost', 'LightGBM', 'MLP (original)', 'MLP (upgraded)', 'Stacking (XGB+LGBM→RF)', 'TabNet']
  特徵數：11 個
  測試集：(1988, 11)
  訓練集：(7948, 11)
  Stage 2 結果：(10, 7)
  三階段對比：10 個模型

三階段 MCC：
  KNN                            Stage1=0.5179  Stage2=0.5361
  Decision Tree                  Stage1=0.7716  Stage2=0.8982
  Random Forest                  Stage1=0.7837  Stage2=0.8910
  Gradient Boosting              Stage1=0.7112  Stage2=0.9247
  XGBoost                        Stage1=0.7406  Stage2=0.9106
  LightGBM                       Stage1=0.8044  Stage2=0.9291
  MLP (original)                 Stage1=0.5765  Stage2=0.7500
  MLP (upgraded)                 Stage1=0.6911  Stage2=0.7958
  Stacking (XGB+LGBM→RF)         Stage1=0.8716  Stage2=0.9163
  TabNet                         Stage1=  —    Stage2=0.7842


In [20]:
print(results2['TabNet']['MCC'])

0.7842369355485763
